<a href="https://colab.research.google.com/github/n9wjntyfw8-max/CAPA-2026-VSNT/blob/main/deteccao_vsnt_modd2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deteccao de embarcacoes em imagens de VSNT (MODD2)

Comparacao entre YOLOv8, YOLO11, YOLO26 e SSD300 na deteccao de embarcacoes e
obstaculos flutuantes, medindo cada modelo antes e depois de adaptar ao dominio
maritimo.

A ideia nao e chegar no detector mais preciso possivel. E colocar as quatro
arquiteturas sob o mesmo protocolo (mesmos dados, mesma particao, mesmas
anotacoes, mesmas metricas) e ver o que o fine-tuning muda em cada uma.

Ordem do notebook:

1. Ambiente: GPU, bibliotecas, pastas
2. Download do MODD2, imagens retificadas e anotacoes oficiais
3. Olhar o conjunto antes de detectar qualquer coisa
4. Separar dois quadros de referencia, um com alvo grande e um com alvo pequeno
5. Rodar os quatro modelos pre-treinados no COCO
6. Converter as anotacoes e montar treino / validacao / teste
7. Avaliar ANTES do fine-tuning
8. Fine-tuning dos quatro
9. Avaliar DEPOIS: matrizes, mAP e tempo de inferencia

A ultima etapa gera `tabela_artigo.csv`, uma linha por modelo, que e o que vai
para o artigo.

## Antes de rodar

Algumas coisas que aprendi na pratica com este conjunto e que vale ter em mente:

**As anotacoes sao oficiais.** O MODD2 vem com ground truth feito a mao e conferido
por especialista: cada obstaculo tem caixa, e a linha d'agua e um poligono. Nenhum
rotulo e gerado aqui dentro. Isso importa porque, se eu gerasse os rotulos com um
dos modelos comparados, a comparacao ficaria viciada a favor dele.

**Tem que usar o par retificado.** As anotacoes existem em duas versoes, para imagem
RAW e para imagem retificada. Se misturar, a caixa sai deslocada em uns 45 px. O
notebook avisa quando detecta essa combinacao, mas confira o ground truth desenhado
na Etapa 6 antes de treinar.

**So a camera esquerda.** O MODD2 anota apenas o lado L. A direita existe no conjunto
e deixei como trabalho futuro, seja para estimar distancia por disparidade, seja para
reduzir falso alarme exigindo concordancia entre as duas vistas.

**A particao e por sequencia, nao por quadro.** Quadros vizinhos sao quase iguais;
dividir por quadro deixaria o modelo colar na validacao. Com tres ou mais sequencias
o notebook separa cenas inteiras para teste, que o modelo nunca viu.

**Espere numeros baixos.** A mediana da caixa e cerca de 0,02% da imagem e uns 92%
dos obstaculos ficam abaixo de 1%. E um problema de alvo pequeno. O SSD300 sofre mais
que os outros porque reduz a entrada para 300x300, o que faz um alvo de 15 px
praticamente desaparecer. Isso e resultado da comparacao, nao defeito do experimento.

**Custo de GPU.** Treinar nos 11.675 quadros levaria horas por modelo. `PASSO_QUADROS`
pega 1 quadro a cada N (uso 5), o que cabe numa sessao do Colab. Como quadro vizinho e
redundante, nao perco informacao relevante.

# Parte 1 — Preparacao do ambiente

## 1.1 — Verificando o ambiente

A primeira coisa a fazer em qualquer projeto de visao computacional e saber em que maquina voce esta.

A celula abaixo mostra:
-Qual GPU o Colab te deu (T4, L4, A100...) e quanta memoria de video ela tem
-A versao do PyTorch e se o CUDA esta ativo
-Espaco em disco e RAM

**O que observar:** se aparecer `CUDA disponivel: False`, volte em *Ambiente de execucao* e ative a GPU. O restante do notebook funciona na CPU, mas o treinamento (Parte 7) pode levar horas em vez de minutos.

In [ ]:
# ------------------------------------------------------------------
# Diagnostico do ambiente Colab
# ------------------------------------------------------------------
import sys, platform, subprocess

print("=" * 70)
print("AMBIENTE DE EXECUCAO")
print("=" * 70)
print(f"Python  : {platform.python_version()}")

try:
    import torch
    print(f"PyTorch : {torch.__version__}")
    print(f"CUDA disponivel: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM    : {total:.1f} GB")
    else:
        print("!! Sem GPU -> Ambiente de execucao > Alterar tipo > GPU (T4)")
except ImportError:
    print("PyTorch ainda nao importado (sera instalado a seguir)")

print("-" * 70)
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

## 1.2 — Instalando as bibliotecas

| Pacote | Para que serve |
|---|---|
| `ultralytics` | Implementacao oficial de YOLOv8, YOLO11 e YOLO26 (mesma API para os tres) |
| `torchvision` | Ja vem no Colab, traz o SSD300-VGG16 pre-treinado no COCO |
| `gdown` (>= 6.x) | Baixar a pasta do Google Drive com as imagens. Versoes antigas do gdown limitavam pastas a 50 arquivos; como o conjunto tem centenas de imagens, e obrigatorio atualizar. |
| `statsmodels` | Teste exato de McNemar, usado na Etapa 9.3 para checar vies entre as cameras L/R |
| `opencv-python` | Ler/escrever imagens e o "sequencia em video", desenhar bounding boxes |

> Importante: usamos `-U` (upgrade) no `ultralytics` porque o YOLO26 so existe em versoes recentes do pacote. Se o `yolo26n.pt` falhar mais adiante, o notebook usa `yolo12n.pt` como alternativa automatica (avisando na tela).

A instalacao leva ~1-2 minutos. Ignore avisos de dependencias do Colab, sao normais.

In [ ]:
# ------------------------------------------------------------------
# Instalacao das dependencias
# ------------------------------------------------------------------
%pip install -q -U ultralytics
%pip install -q -U gdown
%pip install -q statsmodels pyyaml

import importlib, ultralytics
importlib.reload(ultralytics)

print("\n" + "=" * 70)
print(f"ultralytics : {ultralytics.__version__}")
import torch, torchvision, cv2, gdown
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"opencv      : {cv2.__version__}")
print(f"gdown       : {gdown.__version__}")
print("=" * 70)
print("Instalacao concluida.")

## 1.3 — Estrutura de pastas e configuracao central

Boa pratica de projeto: um unico lugar onde todos os caminhos e hiperparametros ficam definidos. Se voce quiser rodar mais rapido (ou mais devagar e melhor), mexe so nesta celula.

Estrutura criada:

```
/content/projeto_modd2/
 dados/          imagens brutas extraidas do zip/Drive
 anotacoes/      caixas da anotacao oficial, conferencia visual do ground truth, labels finais
 dataset/        dataset YOLO final (train/valid + data.yaml)
 saidas/         sequencias em video, graficos, matrizes de confusao
 treinos/        checkpoints dos modelos treinados
```

**Parametros que voce pode ajustar:**
-`EPOCAS` — mais epocas = melhor modelo, mais tempo. O valor abaixo (60) segue o que foi definido para este experimento; reduza para 15-20 se estiver com pouco tempo de GPU.
-`CONF_MIN` — confianca minima para exibir uma deteccao (0.25 e o padrao YOLO)
-`IMGSZ` — resolucao de entrada dos YOLO. Usamos 1024 (maior que o padrao 640) porque a embarcacao distante ocupa poucos pixels, resolucao maior ajuda a preservar esse sinal.
-`CONF_TEACHER` — confianca minima do anotacao oficial ao sugerir candidatos a alvo na Etapa 7 (baixa de proposito, para nao perder o alvo distante; a conferencia visual do ground truth filtra os exageros)
-`LIMIAR_GAP_REINICIO` — se o seu conjunto tiver mais de uma sessao de captura (uma lacuna grande nos indices, ver Etapa 2.2), este e o tamanho da lacuna a partir do qual o notebook trata os dois lados como sequencias independentes: o rastreio do alvo (Etapa 5.5) reinicia do zero, e a particao treino/validacao (Etapa 5.8) nunca forma um bloco que atravesse a lacuna.

In [ ]:
# ------------------------------------------------------------------
# Configuracao central do projeto
# ------------------------------------------------------------------
import os
import torch
from pathlib import Path

RAIZ       = Path("/content/projeto_modd2")
DADOS      = RAIZ / "dados"
DATASET    = RAIZ / "dataset"
SAIDAS     = RAIZ / "saidas"
TREINOS    = RAIZ / "treinos"
for p in (DADOS, DATASET, SAIDAS, TREINOS):
    p.mkdir(parents=True, exist_ok=True)

# ---------- Fonte dos dados (MODD2, imagens retificadas) ----------
# Fontes tentadas nesta ordem, parando na primeira que funcionar:
#   1. PASTA_LOCAL      -> pasta ja extraida, nada e baixado
#   2. DRIVE_FILE_ID    -> copia propria no Google Drive
#   3. URL_IMAGENS/...  -> zips oficiais da ViCoS (padrao)
USAR_DRIVE    = False   # True apenas se quiser salvar checkpoints no proprio Drive
PASTA_LOCAL   = ""
DRIVE_FILE_ID = ""    # preencher so para usar uma copia propria no Drive

URL_IMAGENS   = "https://box.vicos.si/borja/modd2_dataset/MODD2_video_data_rectified.zip"
URL_ANOTACOES = "https://box.vicos.si/borja/modd2_dataset/MODD2_annotations_v2_rectified.zip"

# Sequencias a utilizar. Lista vazia = todas as 28 do conjunto.
# Com menos de 3 sequencias nao e possivel particionar por sequencia
# (o notebook cai automaticamente para particao por blocos de quadros).
SEQUENCIAS_USADAS = []

# Quadros consecutivos do MODD2 sao quase identicos entre si. Usar 1 a cada
# PASSO_QUADROS reduz o custo de treino sem perda real de informacao.
# 1 = todos os 11.675 quadros (varias horas de GPU por modelo)
# 5 = ~2.300 quadros (cabe numa sessao do Colab)
PASSO_QUADROS = 5

LADO_USADO       = "L"     # o MODD2 so anota a camera esquerda
NOME_CLASSE      = "obstacle"

# ---------- Hiperparametros de deteccao ----------
CONF_MIN         = 0.25
IOU_NMS          = 0.45
IMGSZ            = 768     # 1024 e melhor p/ alvo pequeno, mas ~2x mais lento
IOU_AVALIACAO    = 0.50
IOU_AVALIACAO_2  = 0.30    # limiar secundario (ver nota sobre convencao de caixa)
LIMITE_AVALIACAO = 0       # maximo de imagens por avaliacao (0 = todas)

# ---------- Particao ----------
SEMENTE          = 42
PROPORCAO_TREINO = 0.70
PROPORCAO_VALID  = 0.15    # o restante (0.15) vai para teste
TAMANHO_BLOCO    = 10      # usado apenas quando ha menos de 3 sequencias

# ---------- Fine-tuning ----------
EPOCAS   = 20      # objetivo e comparar antes x depois, nao maximizar acuracia
BATCH    = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Raiz do projeto : {RAIZ}")
print(f"Dispositivo     : {DEVICE}")
print(f"Epocas          : {EPOCAS} | Batch: {BATCH} | imgsz: {IMGSZ}")
print(f"Passo de quadros: 1 a cada {PASSO_QUADROS}")
print(f"Particao        : treino {PROPORCAO_TREINO:.0%} | valid {PROPORCAO_VALID:.0%} | teste {1-PROPORCAO_TREINO-PROPORCAO_VALID:.0%}")

## 1.4 — Google Drive: checkpoints e backup de resultados

Roda uma vez, logo no inicio, e autorize o acesso quando pedido.

-Se ja existir um checkpoint de um modelo no Drive, o treino desse
  modelo e pulado automaticamente (Etapas 7.1 e 7.2) -- util se a
  sessao cair no meio e voce precisar continuar sem retreinar tudo.
-`sincronizar_saidas_com_drive()` copia tudo de `saidas/` para o Drive;
  chamada automaticamente apos as etapas que geram resultado novo.

In [ ]:
# ------------------------------------------------------------------
# Backup opcional dos resultados no Google Drive
# ------------------------------------------------------------------
# Desligado por padrao. Sessao do Colab que cai perde o que estiver so
# em /content, entao em treino longo vale ligar USAR_DRIVE na celula de
# configuracao.
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    PASTA_DRIVE        = Path("/content/drive/MyDrive/deteccao_vsnt_modd2")
    PASTA_DRIVE_CKPT   = PASTA_DRIVE / "checkpoints"
    PASTA_DRIVE_SAIDAS = PASTA_DRIVE / "saidas"
    for _p in (PASTA_DRIVE_CKPT, PASTA_DRIVE_SAIDAS):
        _p.mkdir(parents=True, exist_ok=True)
    print("[ok] Drive montado.")
    print(f"  Checkpoints -> {PASTA_DRIVE_CKPT}")
    print(f"  Saidas      -> {PASTA_DRIVE_SAIDAS}")
else:
    PASTA_DRIVE = PASTA_DRIVE_CKPT = PASTA_DRIVE_SAIDAS = None
    print("[info] Drive desligado. Resultados ficam apenas em", SAIDAS)

def sincronizar_saidas_com_drive():
    """Copia SAIDAS para o Drive. Nao faz nada se USAR_DRIVE for False."""
    if not USAR_DRIVE:
        return
    import shutil
    n = 0
    for arq in SAIDAS.iterdir():
        if arq.is_file():
            shutil.copy2(arq, PASTA_DRIVE_SAIDAS / arq.name)
            n += 1
    print(f"[drive] {n} arquivo(s) sincronizado(s) em {PASTA_DRIVE_SAIDAS}")

# Parte 2 — Obtendo o MODD2

## 2.1 — Imagens retificadas e anotacoes oficiais

O notebook aceita duas fontes, escolhidas em `PASTA_LOCAL` na celula de configuracao:

-**vazio**  baixa da ViCoS com `wget` (1,72 GB de imagens + 7 MB de anotacoes, ~2 min)
-**caminho**  usa uma pasta ja baixada (Drive montado ou `/content`), que deve conter
  as subpastas `video_data/` e `annotationsV2_rectified/`

A celula so aceita sequencias que tenham imagem e anotacao, e avisa se o caminho das
anotacoes nao contiver `rectified` enquanto as imagens vierem de `framesRectified`.

In [ ]:
# ------------------------------------------------------------------
# Obtencao do MODD2: imagens retificadas + anotacoes oficiais
# ------------------------------------------------------------------
import zipfile, shutil, subprocess

PASTA_DADOS = DADOS / "modd2"
PASTA_DADOS.mkdir(parents=True, exist_ok=True)

def _achar_subpastas(raiz, nome_alvo):
    """{sequencia: caminho} para toda subpasta chamada nome_alvo, em qualquer nivel.

    Ignora '__MACOSX': zips criados no macOS carregam uma arvore espelho com
    arquivos '._'. Sem esse filtro ela sobrescreve a pasta real no dicionario
    e o notebook acha zero imagens.
    """
    achados = {}
    for p in Path(raiz).rglob(nome_alvo):
        if not p.is_dir() or "__MACOSX" in p.parts:
            continue
        achados[p.parent.name] = p
    return achados

def _ja_extraido(raiz):
    return bool(_achar_subpastas(raiz, "framesRectified")) and bool(_achar_subpastas(raiz, "ground_truth"))

def _extrair(zip_path, destino):
    print(f"Extraindo {Path(zip_path).name} ...", flush=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(destino)

RAIZ_DADOS = None
FONTE = None

# ---- 1. Pasta local ja extraida ----
if PASTA_LOCAL:
    base = Path(PASTA_LOCAL)
    assert base.exists(), f"PASTA_LOCAL nao encontrada: {base}"
    RAIZ_DADOS, FONTE = base, "pasta local"

# ---- 2. DATASET.zip no Google Drive ----
elif DRIVE_FILE_ID and not _ja_extraido(PASTA_DADOS):
    zip_drive = DADOS / "DATASET.zip"
    if not zip_drive.exists():
        print("Baixando DATASET.zip do Google Drive (~1,9 GB)...", flush=True)
        import gdown
        gdown.download(id=DRIVE_FILE_ID, output=str(zip_drive), quiet=False)
    _extrair(zip_drive, PASTA_DADOS)
    try:
        zip_drive.unlink()          # libera ~1,9 GB de disco
    except OSError:
        pass
    RAIZ_DADOS, FONTE = PASTA_DADOS, "Google Drive"

elif DRIVE_FILE_ID:
    RAIZ_DADOS, FONTE = PASTA_DADOS, "Google Drive (ja extraido)"

# ---- 3. Zips oficiais da ViCoS ----
else:
    for url, nome in ((URL_ANOTACOES, "anotacoes.zip"), (URL_IMAGENS, "imagens.zip")):
        alvo = DADOS / nome
        if not alvo.exists():
            print(f"Baixando {nome} ...", flush=True)
            subprocess.run(["wget", "-q", "--show-progress", "-O", str(alvo), url], check=True)
        _extrair(alvo, PASTA_DADOS)
        try:
            alvo.unlink()
        except OSError:
            pass
    RAIZ_DADOS, FONTE = PASTA_DADOS, "ViCoS"

# ---- Descoberta das sequencias ----
seqs_img = _achar_subpastas(RAIZ_DADOS, "framesRectified")
seqs_gt  = _achar_subpastas(RAIZ_DADOS, "ground_truth")

SEQUENCIAS = sorted(set(seqs_img) & set(seqs_gt))
if SEQUENCIAS_USADAS:
    faltando = set(SEQUENCIAS_USADAS) - set(SEQUENCIAS)
    SEQUENCIAS = [s for s in SEQUENCIAS if s in SEQUENCIAS_USADAS]
    if faltando:
        print(f"[aviso] pedidas mas indisponiveis: {sorted(faltando)}")

assert SEQUENCIAS, (
    f"Nenhuma sequencia com imagem E anotacao em {RAIZ_DADOS}.\n"
    f"  pastas 'framesRectified': {len(seqs_img)}\n"
    f"  pastas 'ground_truth'   : {len(seqs_gt)}\n"
    "Verifique se o pacote contem as duas coisas e se ambas sao da versao RETIFICADA."
)

# ---- Guarda: anotacao RAW sobre imagem retificada desloca a caixa em ate ~45 px ----
if FONTE != "ViCoS" and "rectif" not in str(seqs_gt[SEQUENCIAS[0]]).lower():
    print("\n" + "!" * 70)
    print("AVISO: imagens sao RETIFICADAS, mas o caminho das anotacoes nao contem")
    print(f"'rectified': {seqs_gt[SEQUENCIAS[0]]}")
    print("Se forem as anotacoes RAW, as caixas ficarao deslocadas ~45 px.")
    print("Confira o ground truth desenhado na Etapa 6 antes de treinar.")
    print("!" * 70 + "\n")

CAMINHO_SEQ = {s: seqs_img[s] for s in SEQUENCIAS}
CAMINHO_GT  = {s: seqs_gt[s]  for s in SEQUENCIAS}

print(f"\n[ok] fonte: {FONTE}")
print(f"[ok] {len(SEQUENCIAS)} sequencia(s):")
for s in SEQUENCIAS:
    n = len([p for p in CAMINHO_SEQ[s].glob(f"*{LADO_USADO}.jpg") if not p.name.startswith("._")])
    print(f"   {s:42s} {n:5d} imagens {LADO_USADO}")

## 2.2 — Indexando os quadros

Os nomes seguem o padrao `NNNNNNNNL.jpg` — indice de captura de 8 digitos mais a camera.
A celula abaixo percorre as pastas de sequencia e monta uma tabela com uma linha por quadro,
guardando de qual sequencia ele veio, informacao usada depois para particionar por cena.

Quadros sem anotacao correspondente sao descartados e contabilizados.

**O que observar:** quantos quadros sobraram por sequencia e se algum foi descartado. Um numero
alto de descartes indica que imagens e anotacoes vieram de versoes diferentes (RAW x retificada).

In [ ]:
# ------------------------------------------------------------------
# Indexacao dos quadros: so entra quadro que tem imagem E anotacao
# ------------------------------------------------------------------
import re
import cv2
import numpy as np
import pandas as pd

PADRAO_NOME = re.compile(r"(\d+)([LR])\.jpe?g$", re.IGNORECASE)

def carregar_rgb(caminho):
    """Le uma imagem do disco e devolve em RGB (o OpenCV le em BGR)."""
    bgr = cv2.imread(str(caminho))
    if bgr is None:
        raise FileNotFoundError(f"Nao consegui ler a imagem: {caminho}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

registros = []
sem_par = 0
for seq in SEQUENCIAS:
    for caminho in sorted(CAMINHO_SEQ[seq].glob(f"*{LADO_USADO}.jp*g")):
        if caminho.name.startswith("._"):
            continue
        m = PADRAO_NOME.search(caminho.name)
        if not m:
            continue
        mat = CAMINHO_GT[seq] / (caminho.stem + ".mat")
        if not mat.exists():          # anotacao ausente -> quadro descartado
            sem_par += 1
            continue
        registros.append({"sequencia": seq, "arquivo": caminho.name,
                          "caminho": str(caminho), "mat": str(mat),
                          "indice": int(m.group(1))})

if not registros:
    linhas = ["Nenhum quadro com imagem E anotacao correspondente.", ""]
    for seq in SEQUENCIAS[:5]:
        jpgs = [p for p in CAMINHO_SEQ[seq].glob("*.jp*g") if not p.name.startswith("._")]
        mats = [p for p in CAMINHO_GT[seq].glob("*.mat")   if not p.name.startswith("._")]
        linhas += [f"  {seq}",
                   f"     imagens : {len(jpgs)} em {CAMINHO_SEQ[seq]}",
                   f"     exemplo : {jpgs[0].name if jpgs else '(nenhuma)'}",
                   f"     anotacao: {len(mats)} em {CAMINHO_GT[seq]}",
                   f"     exemplo : {mats[0].name if mats else '(nenhuma)'}"]
    linhas += ["", "Causas comuns:",
               "  - pasta '__MACOSX' do zip mascarando a pasta real",
               "  - imagens e anotacoes de versoes diferentes (RAW x retificada)",
               "  - nomes fora do padrao NNNNNNNNL.jpg"]
    raise RuntimeError("\n".join(linhas))

df_imgs = pd.DataFrame(registros).sort_values(["sequencia", "indice"]).reset_index(drop=True)

# subamostragem: 1 quadro a cada PASSO_QUADROS, dentro de cada sequencia
if PASSO_QUADROS > 1:
    n_antes = len(df_imgs)
    manter = df_imgs.groupby("sequencia").cumcount() % PASSO_QUADROS == 0
    df_imgs = df_imgs[manter].reset_index(drop=True)
    print(f"[subamostragem] 1 a cada {PASSO_QUADROS} quadros: {n_antes} -> {len(df_imgs)}")
assert len(df_imgs) > 0, "Nenhum quadro com imagem e anotacao correspondentes."

print("=" * 70)
print("QUADROS INDEXADOS (imagem + anotacao oficial)")
print("=" * 70)
print(df_imgs.groupby("sequencia").size().to_string())
print("-" * 70)
print(f"Total de quadros utilizaveis : {len(df_imgs)}")
print(f"Descartados por falta de par : {sem_par}")

amostra = cv2.imread(df_imgs.iloc[0]["caminho"])
ALTURA_IMG, LARGURA_IMG = amostra.shape[:2]
print(f"Resolucao                    : {LARGURA_IMG} x {ALTURA_IMG}")
print("=" * 70)

### 2.2.1 — Nota sobre o par estereo (o que ele significa fisicamente)

As cameras `L` e `R` estao retificadas e apontam no mesmo sentido, com uma pequena distancia entre elas (a *baseline*). Isso tem duas consequencias uteis para este trabalho:

-A mesma cena, no mesmo instante, e vista de dois pontos ligeiramente diferentes, o alvo aparece deslocado horizontalmente entre as duas imagens (a *disparidade*, que codifica distancia), mas praticamente na mesma altura (retificacao vertical).
-Por isso, um par L/R funciona como um experimento controlado: mesma embarcacao, mesma distancia aproximada, mesma iluminacao, unica variavel real e qual camera gerou a imagem. Se um detector encontra a embarcacao em `L` e nao em `R` (ou vice-versa) para o mesmo instante, essa discordancia e atribuivel ao detector, nao a cena.

Guardamos essa ideia para a Etapa 9.3, onde ela vira uma analise quantitativa (consistencia entre cameras) que voce pode citar diretamente no artigo.

## 2.3 — Observando o conjunto de imagens ANTES de detectar

> Esta e uma das etapas mais importantes e mais ignoradas.

Antes de jogar qualquer modelo no problema, olhe os dados com seus proprios olhos. Como nao ha um `.mp4` para assistir, fazemos duas coisas equivalentes:

1. Um mosaico com 16 imagens (lado `L`) igualmente espacadas ao longo da sequencia.
2. Um sequencia em video: as imagens do lado `L`, ordenadas por indice, viram frames de um `.mp4` gerado na hora, so para navegacao visual rapida. Para nao gerar falsa impressao: isso nao e um video de verdade, e sim uma sequencia de fotos tocada em ordem.

**O que voce precisa responder enquanto olha:**
1. Que outros objetos aparecem na cena (outras embarcacoes, linha de costa, reflexo na agua)?
2. Como a embarcacao aparece, grande e nitida em alguns trechos, um ponto pequeno em outros?
3. Ha variacao de iluminacao ou de estado do mar ao longo da sequencia?

**Anote mentalmente** (ou em papel) um indice em que a embarcacao esta bem perto da camera e outro em que ela esta bem longe, vamos fixar esses dois na Etapa 2.4.

In [ ]:
# ------------------------------------------------------------------
# Mosaico com 16 imagens (lado L) igualmente espacadas
# ------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

arquivos_L = df_imgs.sort_values(["sequencia", "indice"]).reset_index(drop=True)

N_MOSAICO = 16
n = min(N_MOSAICO, len(arquivos_L))
posicoes = np.linspace(0, len(arquivos_L) - 1, n).astype(int)

fig, axes = plt.subplots(4, 4, figsize=(18, 11))
for ax, pos in zip(axes.ravel(), posicoes):
    linha = arquivos_L.iloc[pos]
    img = np.array(Image.open(linha["caminho"]).convert("RGB"))
    ax.imshow(img)
    ax.set_title(f"idx {linha['indice']}  ({linha['arquivo']})", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[n:]:
    ax.axis("off")

plt.suptitle("Mosaico (camera L) — procure uma imagem com a alvo GRANDE e outra LONGE",
            fontsize=14, y=0.995)
plt.tight_layout()
plt.savefig(SAIDAS / "mosaico_imagens.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# ------------------------------------------------------------------
# "Video sintetico" a partir da sequencia ordenada de imagens (lado L)
# ------------------------------------------------------------------
from IPython.display import HTML, display
from base64 import b64encode
import subprocess

def comprimir_para_navegador(entrada, saida, largura=640, crf=28, segundos=None):
    """Reencoda para H.264 baseline, que todo navegador consegue tocar."""
    cmd = ["ffmpeg", "-y", "-i", str(entrada)]
    if segundos:
        cmd += ["-t", str(segundos)]
    cmd += ["-vf", f"scale={largura}:-2", "-vcodec", "libx264", "-preset", "veryfast",
            "-crf", str(crf), "-pix_fmt", "yuv420p", "-an", "-loglevel", "error", str(saida)]
    subprocess.run(cmd, check=True)
    return saida

def mostrar_video(caminho, largura=720, titulo=""):
    """Exibe um mp4 embutido em base64 (funciona offline no Colab)."""
    caminho = Path(caminho)
    mb = caminho.stat().st_size / 1e6
    if mb > 45:
        print(f"[aviso] {caminho.name} tem {mb:.0f} MB - comprimindo para exibir...")
        prev = caminho.with_name(caminho.stem + "_preview.mp4")
        comprimir_para_navegador(caminho, prev, largura=480, crf=32)
        caminho = prev
    b64 = b64encode(caminho.read_bytes()).decode()
    if titulo:
        display(HTML(f"<h4 style='font-family:sans-serif'>{titulo}</h4>"))
    display(HTML(f"""
    <video width={largura} controls loop style="border-radius:8px;border:1px solid #ccc">
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>"""))

def montar_video_sintetico(lista_caminhos, caminho_saida, fps=6):
    """Concatena uma sequencia de imagens em um .mp4, na ordem dada."""
    primeira = cv2.imread(str(lista_caminhos[0]))
    h, w = primeira.shape[:2]
    temporario = str(Path(caminho_saida).with_suffix(".tmp.mp4"))
    escritor = cv2.VideoWriter(temporario, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for caminho in lista_caminhos:
        frame = cv2.imread(str(caminho))
        if frame is not None:
            escritor.write(frame)
    escritor.release()
    comprimir_para_navegador(temporario, caminho_saida, largura=640, crf=28)
    os.remove(temporario)
    return caminho_saida

FPS_SINTETICO = 6   # so para visualizacao -- nao corresponde a uma taxa de captura real
VIDEO_SINTETICO_L = SAIDAS / "sequencia_L_sintetica.mp4"
if not VIDEO_SINTETICO_L.exists():
    print("Montando video sintetico a partir das imagens (lado L)...")
    montar_video_sintetico(list(arquivos_L["caminho"]), VIDEO_SINTETICO_L, fps=FPS_SINTETICO)

mostrar_video(VIDEO_SINTETICO_L, titulo=" Sequencia sintetica (camera L) — nao e um video real, so a ordem de captura")

## 2.4 — Dois quadros de referencia (alvo GRANDE / alvo PEQUENO)

Aqui fixamos dois quadros usados em todos os experimentos, antes e depois do treinamento.
Usar sempre os mesmos e o que torna a comparacao justa: mudou o resultado? Foi o modelo,
nao a imagem.

A escolha e automatica, pela area da maior caixa anotada: o quadro com o maior obstaculo
representa o caso facil, e um dos 5% menores representa o caso dificil. Nao ha nada para editar.

In [ ]:
# ------------------------------------------------------------------
# Dois quadros de referencia: alvo GRANDE (facil) e alvo PEQUENO (dificil)
# Escolha automatica pela area da maior caixa anotada, com o ground truth
# desenhado e um recorte ampliado -- sem isso o alvo pequeno e invisivel.
# ------------------------------------------------------------------
import scipy.io as sio

LADO_MIN_REF = 14   # px: evita escolher como referencia um alvo indistinguivel

def _caixas_do_mat(caminho_mat):
    ob = sio.loadmat(str(caminho_mat))["annotations"][0, 0]["obstacles"]
    if not ob.size:
        return []
    return [tuple(map(float, l[:4])) for l in np.atleast_2d(np.asarray(ob, dtype=np.float64))]

df_ref = df_imgs.copy()
caixas_por_quadro = {r["caminho"]: _caixas_do_mat(r["mat"]) for _, r in df_ref.iterrows()}
df_ref["area"] = [max((w * h for _, _, w, h in caixas_por_quadro[c]), default=0.0)
                  for c in df_ref["caminho"]]
df_ref["lado_min"] = [max((min(w, h) for _, _, w, h in caixas_por_quadro[c]), default=0.0)
                      for c in df_ref["caminho"]]

com_alvo = df_ref[df_ref["area"] > 0].sort_values("area")
visiveis = com_alvo[com_alvo["lado_min"] >= LADO_MIN_REF]
if len(visiveis) < 2:
    visiveis = com_alvo          # conjunto extremo: aceita o que houver

LINHA_PERTO = com_alvo.iloc[-1]                        # maior alvo do conjunto
LINHA_LONGE = visiveis.iloc[0]                         # menor alvo ainda distinguivel

CAMINHO_PERTO, CAMINHO_LONGE = LINHA_PERTO["caminho"], LINHA_LONGE["caminho"]
img_perto, img_longe = carregar_rgb(CAMINHO_PERTO), carregar_rgb(CAMINHO_LONGE)
NOME_PERTO, NOME_LONGE = LINHA_PERTO["arquivo"], LINHA_LONGE["arquivo"]

def _com_gt(img, caixas):
    """Copia da imagem com o ground truth desenhado em verde."""
    out = img.copy()
    esp = max(2, img.shape[1] // 400)
    for (x, y, w, h) in caixas:
        cv2.rectangle(out, (int(x), int(y)), (int(x + w), int(y + h)), (0, 255, 0), esp)
    return out

def _recorte(img, caixas, margem=90):
    """Recorte ampliado em volta da maior caixa, para o alvo pequeno ficar visivel."""
    if not caixas:
        return None
    x, y, w, h = max(caixas, key=lambda c: c[2] * c[3])
    cx, cy = x + w / 2, y + h / 2
    H, W = img.shape[:2]
    x0, x1 = int(max(0, cx - margem)), int(min(W, cx + margem))
    y0, y1 = int(max(0, cy - margem)), int(min(H, cy + margem))
    rec = img[y0:y1, x0:x1].copy()
    cv2.rectangle(rec, (int(x - x0), int(y - y0)), (int(x + w - x0), int(y + h - y0)), (0, 255, 0), 2)
    return cv2.resize(rec, (420, 420), interpolation=cv2.INTER_NEAREST)

cx_perto, cx_longe = caixas_por_quadro[CAMINHO_PERTO], caixas_por_quadro[CAMINHO_LONGE]

fig, axes = plt.subplots(2, 2, figsize=(17, 11),
                         gridspec_kw={"width_ratios": [2.6, 1]})
for lin, (img, caixas, nome, linha, rot) in enumerate((
        (img_perto, cx_perto, NOME_PERTO, LINHA_PERTO, "ALVO GRANDE"),
        (img_longe, cx_longe, NOME_LONGE, LINHA_LONGE, "ALVO PEQUENO"))):
    px = 100 * linha["area"] / (LARGURA_IMG * ALTURA_IMG)
    axes[lin, 0].imshow(_com_gt(img, caixas))
    axes[lin, 0].set_title(f"{rot} — {nome} — {len(caixas)} obstaculo(s) — "
                           f"maior = {px:.3f}% da imagem", fontsize=12)
    axes[lin, 0].axis("off")
    rec = _recorte(img, caixas)
    if rec is not None:
        axes[lin, 1].imshow(rec)
        axes[lin, 1].set_title("recorte ampliado", fontsize=11)
    axes[lin, 1].axis("off")
plt.suptitle("Quadros de referencia — ground truth em verde", fontsize=15)
plt.tight_layout()
plt.savefig(SAIDAS / "quadros_referencia.png", dpi=110, bbox_inches="tight")
plt.show()

for rot, linha, caixas in (("PERTO", LINHA_PERTO, cx_perto), ("LONGE", LINHA_LONGE, cx_longe)):
    _, _, w, h = max(caixas, key=lambda c: c[2] * c[3])
    print(f"Referencia {rot}: {linha['arquivo']}  ({linha['sequencia']})")
    print(f"   maior caixa: {w:.0f} x {h:.0f} px "
          f"({100 * w * h / (LARGURA_IMG * ALTURA_IMG):.3f}% da imagem) | "
          f"{len(caixas)} obstaculo(s) no quadro")

# Parte 3 — Os quatro detectores nas imagens de referencia

## 3.1 — Carregando YOLOv8, YOLO11, YOLO26 e SSD

Vamos carregar quatro detectores pre-treinados no dataset COCO (80 classes: pessoa, carro, aviao, passaro, barco (`boat`), etc.).

### As arquiteturas, em uma frase cada

| Modelo | Ano | Ideia central |
|---|---|---|
| **YOLOv8** | 2023 | Anchor-free, cabeca desacoplada (classificacao e regressao separadas), NMS na saida |
| **YOLO11** | 2024 | Backbone refinado (C3k2, C2PSA com atencao), mais preciso com menos parametros |
| **YOLO26** | 2026 | End-to-end nativo (sem NMS), remove o DFL, treino com MuSGD + ProgLoss + STAL (foco em objetos pequenos) |
| **SSD300** | 2016 | *Single Shot Detector* classico: ancoras fixas em multiplos mapas de caracteristicas, backbone VGG16 |

O SSD entra como linha de base historica: mostra de onde a area veio e por que os YOLO modernos sao tao melhores em objetos pequenos.

> Usamos as variantes nano (`n`) dos YOLO — sao as menores e mais rapidas. O `yolo26n.pt` e tentado primeiro; se a versao do `ultralytics` instalada ainda nao o tiver, a celula cai automaticamente para `yolo12n.pt` e avisa na tela, so por seguranca, o metodo continua o mesmo.

Os pesos sao baixados automaticamente na primeira execucao (~50 MB no total).

In [ ]:
# ------------------------------------------------------------------
# Carregando os 4 detectores pre-treinados no COCO
# ------------------------------------------------------------------
from ultralytics import YOLO
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

modelos = {}
PESOS_USADOS = {}   # nome -> arquivo de pesos efetivamente carregado (guardado para o fine-tuning na Parte 7)

# ---- Familia YOLO (mesma API para as 3 versoes) ----
pesos_desejados = [("YOLOv8", "yolov8n.pt"), ("YOLO11", "yolo11n.pt"), ("YOLO26", "yolo26n.pt")]
for nome, pesos in pesos_desejados:
    try:
        modelos[nome] = YOLO(pesos)
        PESOS_USADOS[nome] = pesos
        print(f"[ok] {nome:8s} <- {pesos}")
    except Exception as e:
        # NAO ha fallback silencioso. Publicar numero do yolo12 sob o nome
        # "YOLO26" invalidaria a comparacao, que e o objetivo do trabalho.
        raise RuntimeError(
            f"Falha ao carregar {pesos} para {nome}: {e}\n"
            "  -> rode '%pip install -U ultralytics', reinicie o ambiente e tente de novo.\n"
            "  -> se o modelo realmente nao existir nesta versao, remova-o de\n"
            "     'pesos_desejados' e declare a ausencia no artigo. NAO substitua\n"
            "     por outro modelo mantendo o nome."
        ) from e

# ---- SSD300 do torchvision ----
pesos_ssd = SSD300_VGG16_Weights.COCO_V1
ssd = ssd300_vgg16(weights=pesos_ssd).eval().to(DEVICE)
transformacao_ssd = pesos_ssd.transforms()
CLASSES_SSD = pesos_ssd.meta["categories"]   # 91 entradas do COCO (com '__background__' e 'N/A')
print(f"[ok] SSD300  <- VGG16 / COCO ({len([c for c in CLASSES_SSD if c not in ('__background__','N/A')])} classes uteis)")

# ---- Quantos parametros tem cada um? ----
print("\n" + "=" * 70)
print(f"{'MODELO':10s} {'PARAMETROS':>14s}   {'CLASSES':>8s}")
print("=" * 70)
for nome, m in modelos.items():
    n_par = sum(p.numel() for p in m.model.parameters())
    print(f"{nome:10s} {n_par:>14,}   {len(m.names):>8d}")
print(f"{'SSD300':10s} {sum(p.numel() for p in ssd.parameters()):>14,}   {len(CLASSES_SSD):>8d}")
print("=" * 70)

## 3.2 — Uma interface unica para os quatro modelos

Cada biblioteca devolve as deteccoes em um formato diferente. Para conseguir comparar macas com macas, escrevemos funcoes que normalizam tudo para o mesmo formato:

```python
[{"caixa": (x1, y1, x2, y2), "classe": "boat", "conf": 0.87}, ...]
```

Tambem criamos `desenhar_deteccoes()`, que desenha na imagem:
-a bounding box (cor deterministica por classe, a mesma classe sempre tem a mesma cor)
-o label e o confidence level em percentual

### Sobre o *confidence level*
E a probabilidade que o modelo atribui aquela deteccao. `CONF_MIN = 0.25` significa "ignore tudo abaixo de 25%". Baixar esse limiar aumenta o recall (acha mais coisas) e derruba a precisao (acha mais lixo). E o trade-off fundamental da deteccao.

In [ ]:
# ------------------------------------------------------------------
# Interface unificada de deteccao + desenho das bounding boxes
# ------------------------------------------------------------------
import hashlib
import torch

def _cor_da_classe(nome_classe):
    """Cor BGR estavel e distinta para cada nome de classe."""
    h = int(hashlib.md5(nome_classe.encode()).hexdigest()[:6], 16)
    return (60 + (h & 0xFF) % 195, 60 + ((h >> 8) & 0xFF) % 195, 60 + ((h >> 16) & 0xFF) % 195)

def detectar_yolo(modelo, imagem_rgb, conf=CONF_MIN, iou=IOU_NMS, imgsz=None):
    """Roda um modelo Ultralytics e devolve a lista normalizada de deteccoes."""
    r = modelo.predict(imagem_rgb[:, :, ::-1], conf=conf, iou=iou, imgsz=imgsz or IMGSZ,
                       device=DEVICE, verbose=False)[0]
    saida = []
    for caixa in r.boxes:
        x1, y1, x2, y2 = caixa.xyxy[0].tolist()
        saida.append({"caixa": (x1, y1, x2, y2),
                      "classe": r.names[int(caixa.cls[0])],
                      "conf": float(caixa.conf[0])})
    return saida

@torch.no_grad()
def detectar_ssd(imagem_rgb, conf=CONF_MIN, modelo=None, nomes_classes=None):
    """Roda o SSD do torchvision e devolve a lista normalizada de deteccoes."""
    modelo = modelo if modelo is not None else ssd
    nomes_classes = nomes_classes if nomes_classes is not None else CLASSES_SSD
    tensor = torch.from_numpy(imagem_rgb.copy()).permute(2, 0, 1).float() / 255.0
    pred = modelo([tensor.to(DEVICE)])[0]
    saida = []
    for caixa, rotulo, escore in zip(pred["boxes"], pred["labels"], pred["scores"]):
        s = float(escore)
        if s < conf:
            continue
        idx = int(rotulo)
        nome = nomes_classes[idx] if idx < len(nomes_classes) else str(idx)
        if nome in ("__background__", "N/A"):
            continue
        saida.append({"caixa": tuple(caixa.tolist()), "classe": nome, "conf": s})
    return saida

def desenhar_deteccoes(imagem_rgb, deteccoes, espessura=None, escala_texto=None):
    """Desenha caixas + label + confidence sobre uma copia da imagem."""
    img = imagem_rgb.copy()
    h, w = img.shape[:2]
    espessura = espessura or max(2, int(round(w / 700)))
    escala_texto = escala_texto or max(0.5, w / 1600)
    for d in deteccoes:
        x1, y1, x2, y2 = [int(v) for v in d["caixa"]]
        cor = _cor_da_classe(d["classe"])
        cv2.rectangle(img, (x1, y1), (x2, y2), cor, espessura)
        texto = f'{d["classe"]} {d["conf"]*100:.0f}%'
        (tw, th), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, escala_texto, espessura)
        y_topo = max(y1 - th - base - 3, 0)
        cv2.rectangle(img, (x1, y_topo), (x1 + tw + 4, y_topo + th + base + 3), cor, -1)
        cv2.putText(img, texto, (x1 + 2, y_topo + th + 1),
                    cv2.FONT_HERSHEY_SIMPLEX, escala_texto, (255, 255, 255), espessura, cv2.LINE_AA)
    return img

def detectar_com_todos(imagem_rgb, conf=CONF_MIN, dic_modelos=None, ssd_modelo=None, ssd_classes=None):
    """Roda os 4 detectores na mesma imagem e mede o tempo de cada um."""
    import time
    dic_modelos = dic_modelos if dic_modelos is not None else modelos
    resultado = {}
    for nome, m in dic_modelos.items():
        t0 = time.time()
        det = detectar_yolo(m, imagem_rgb, conf=conf)
        resultado[nome] = {"det": det, "ms": (time.time() - t0) * 1000}
    t0 = time.time()
    resultado["SSD300"] = {"det": detectar_ssd(imagem_rgb, conf=conf, modelo=ssd_modelo, nomes_classes=ssd_classes),
                           "ms": (time.time() - t0) * 1000}
    return resultado

print("Funcoes prontas: detectar_yolo, detectar_ssd, desenhar_deteccoes, detectar_com_todos")

## 3.3 — Teste 1: a imagem com a alvo GRANDE

Agora o primeiro experimento de verdade. Rodamos os quatro detectores na mesma imagem e colocamos os resultados lado a lado.

**O que observar no painel:**

1. **Quantas caixas cada modelo desenhou?** Modelos mais novos costumam ser mais "confiantes" e menos ruidosos.
2. **A classe `boat` aparece?** E bem provavel que sim, mas repare quantas caixas de `boat` aparecem: se houver mais de uma, o modelo esta vendo outras embarcacoes na cena, nao so a suo obstaculo. Guarde esse numero, e o argumento central da Parte 5.
3. **Os valores de confianca.**
4. **O tempo de inferencia (ms)** impresso na tabela — YOLO26 dispensa NMS, o que reduz o custo de pos-processamento.

In [ ]:
# ------------------------------------------------------------------
# Comparacao dos 4 detectores na imagem com a ALVO GRANDE
# ------------------------------------------------------------------
resultado_perto = detectar_com_todos(img_perto)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
for ax, (nome, r) in zip(axes.ravel(), resultado_perto.items()):
    ax.imshow(desenhar_deteccoes(img_perto, r["det"]))
    ax.set_title(f'{nome}  —  {len(r["det"])} objetos  |  {r["ms"]:.0f} ms', fontsize=14)
    ax.axis("off")
plt.suptitle(f" ALVO GRANDE — {NOME_PERTO} — modelos pre-treinados no COCO",
             fontsize=17, y=0.99)
plt.tight_layout()
plt.savefig(SAIDAS / "comparacao_perto_antes.png", dpi=110, bbox_inches="tight")
plt.show()

# ---- Tabela detalhada ----
linhas = []
for nome, r in resultado_perto.items():
    if not r["det"]:
        linhas.append({"Modelo": nome, "Classe": "(nenhuma deteccao)", "Qtd": 0,
                       "Conf. max": "-", "Tempo (ms)": round(r["ms"])})
        continue
    df = pd.DataFrame(r["det"])
    for classe, g in df.groupby("classe"):
        linhas.append({"Modelo": nome, "Classe": classe, "Qtd": len(g),
                       "Conf. max": f"{g['conf'].max()*100:.1f}%", "Tempo (ms)": round(r["ms"])})

tabela_perto = pd.DataFrame(linhas)
print("\nDETECCOES NA IMAGEM 'ALVO GRANDE':")
display(tabela_perto)

n_boats_perto = {nome: sum(1 for d in r["det"] if d["classe"] == "boat") for nome, r in resultado_perto.items()}
print("\nQuantas caixas 'boat' cada modelo encontrou (pode incluir embarcacoes de fundo):")
for nome, n in n_boats_perto.items():
    print(f"  {nome:8s}: {n} caixa(s) 'boat'")

## 3.4 — Teste 2: a imagem com a alvo PEQUENO

Mesmo experimento, agora no caso dificil. Este e o teste que realmente diferencia os modelos.

**O que esperar:**
-Muito menos deteccoes em geral
-A embarcacao pode nao ser detectada por ninguem, ou aparecer com confianca muito baixa
-O SSD300 tende a ser o pior aqui: ele redimensiona tudo para 300×300 pixels, o que destroi objetos pequenos.
-YOLO26 tem o STAL (*Small-Target-Aware Label Assignment*), pensado exatamente para este cenario, vale observar se ele se sai melhor.

> Experimento sugerido: rode esta celula de novo com `conf=0.10` na chamada de `detectar_com_todos` e veja quantas deteccoes fantasma aparecem. E a ilustracao perfeita do trade-off precisao × recall.

In [ ]:
# ------------------------------------------------------------------
# Comparacao dos 4 detectores na imagem com a ALVO PEQUENO
# ------------------------------------------------------------------
resultado_longe = detectar_com_todos(img_longe)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
for ax, (nome, r) in zip(axes.ravel(), resultado_longe.items()):
    ax.imshow(desenhar_deteccoes(img_longe, r["det"]))
    ax.set_title(f'{nome}  —  {len(r["det"])} objetos  |  {r["ms"]:.0f} ms', fontsize=14)
    ax.axis("off")
plt.suptitle(f" ALVO PEQUENO — {NOME_LONGE} — modelos pre-treinados no COCO",
             fontsize=17, y=0.99)
plt.tight_layout()
plt.savefig(SAIDAS / "comparacao_longe_antes.png", dpi=110, bbox_inches="tight")
plt.show()

linhas = []
for nome, r in resultado_longe.items():
    if not r["det"]:
        linhas.append({"Modelo": nome, "Classe": "(nenhuma deteccao)", "Qtd": 0, "Conf. max": "-"})
        continue
    df = pd.DataFrame(r["det"])
    for classe, g in df.groupby("classe"):
        linhas.append({"Modelo": nome, "Classe": classe, "Qtd": len(g),
                       "Conf. max": f"{g['conf'].max()*100:.1f}%"})

print("\nDETECCOES NA IMAGEM 'ALVO PEQUENO':")
display(pd.DataFrame(linhas))

print("\nRESUMO — total de objetos detectados:")
for nome in resultado_perto:
    print(f"  {nome:8s}  perto: {len(resultado_perto[nome]['det']):3d}   "
          f"longe: {len(resultado_longe[nome]['det']):3d}")

# Parte 4 — Os quatro detectores no conjunto completo de imagens

## 4.1 — Rodando os detectores em toda a sequencia

Imagens isoladas mostram *o que* acontece num instante; a sequencia inteira mostra a estabilidade do detector ao longo do tempo. Um bom detector mantem a caixa firme de um indice para o proximo; um detector instavel faz a caixa "piscar" — aparece, some, reaparece.

Como os dados sao imagens paradas, nao ha uma taxa de captura fixa entre indices consecutivos, sao fotos sequenciais, nao um video com FPS conhecido. Por isso tratamos "estabilidade" com a cautela devida: olhamos a sequencia de indices do lado L, na ordem de captura, e montamos um `.mp4` sintetico anotado, o mesmo recurso da Etapa 2.3, agora com as deteccoes desenhadas. E uma aproximacao valida para inspecao visual de flicker, mas nao equivale a uma analise temporal calibrada em segundos.

A funcao `anotar_sequencia()` abaixo:
1. Percorre as imagens do lado L em ordem de indice
2. Roda o detector escolhido em cada uma
3. Desenha as caixas + labels + confianca
4. Escreve um `.mp4` novo e o reencoda em H.264 para tocar no navegador

 Custo: processamos todas as imagens do lado L (nao ha um limite artificial de "frames" aqui, ja que o conjunto e pequeno o bastante para caber no tempo do notebook).

In [ ]:
# ------------------------------------------------------------------
# Anotacao da sequencia completa de imagens com um detector qualquer
# ------------------------------------------------------------------
import time

def anotar_sequencia(lista_caminhos, caminho_saida, funcao_detectar, rotulo=""):
    """Aplica um detector em cada imagem da sequencia e grava um .mp4 anotado."""
    primeira = cv2.imread(str(lista_caminhos[0]))
    h, w = primeira.shape[:2]
    temporario = str(Path(caminho_saida).with_suffix(".tmp.mp4"))
    escritor = cv2.VideoWriter(temporario, cv2.VideoWriter_fourcc(*"mp4v"), FPS_SINTETICO, (w, h))

    processados, total_deteccoes = 0, 0
    t0 = time.time()
    for i, caminho in enumerate(lista_caminhos):
        bgr = cv2.imread(str(caminho))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        dets = funcao_detectar(rgb)
        total_deteccoes += len(dets)
        anotado = desenhar_deteccoes(rgb, dets)
        cv2.putText(anotado, f"{rotulo} | img {i} | {len(dets)} obj",
                    (12, 34), cv2.FONT_HERSHEY_SIMPLEX, max(0.7, w/1800),
                    (255, 255, 255), 2, cv2.LINE_AA)
        escritor.write(cv2.cvtColor(anotado, cv2.COLOR_RGB2BGR))
        processados += 1
        if processados % 50 == 0:
            print(f"    {rotulo}: {processados} imagens...", flush=True)

    escritor.release()
    comprimir_para_navegador(temporario, caminho_saida, largura=720, crf=26)
    os.remove(temporario)
    dt = time.time() - t0
    print(f"  [{rotulo}] {processados} imagens em {dt:.1f}s "
          f"| media {total_deteccoes/max(processados,1):.1f} obj/imagem")
    return caminho_saida

caminhos_L_ordenados = list(arquivos_L["caminho"])

# --------- Roda os 4 modelos na sequencia inteira ---------
videos_antes = {}

for nome, m in modelos.items():
    saida = SAIDAS / f"sequencia_{nome}_antes.mp4"
    videos_antes[nome] = anotar_sequencia(caminhos_L_ordenados, saida,
                                          lambda rgb, mm=m: detectar_yolo(mm, rgb),
                                          rotulo=nome)

videos_antes["SSD300"] = anotar_sequencia(caminhos_L_ordenados, SAIDAS / "sequencia_SSD300_antes.mp4",
                                          lambda rgb: detectar_ssd(rgb),
                                          rotulo="SSD300")
print("\nTodas as sequencias anotadas.")

## 4.2 — Assistindo as quatro sequencias anotadas

As quatro sequencias abaixo mostram exatamente o mesmo conjunto de imagens, processado por detectores diferentes.

**Roteiro de observacao:**
- Consistencia: a caixa acompanha a embarcacao suavemente ou pisca entre indices vizinhos?
- Coerencia do rotulo: o mesmo objeto muda de classe entre indices consecutivos?
- Falsos positivos: caixas em outras embarcacoes, na linha de costa, em reflexos na agua
- A embarcacao: em que faixa de indices ela deixa de ser detectada (fica pequena demais) e se ha mais de uma caixa `boat` competindo por atencao

In [ ]:
# ------------------------------------------------------------------
# Exibindo as sequencias anotadas pelos modelos pre-treinados
# ------------------------------------------------------------------
for nome, caminho in videos_antes.items():
    mostrar_video(caminho, largura=720, titulo=f" {nome} — pre-treinado no COCO (sequencia sintetica)")

# Parte 5 — O que os modelos ja sabem: a classe `boat`

## 5.1 — Verificando o vocabulario do COCO

Antes de falar em adaptacao, vale confirmar o que os modelos ja sabem. Os quatro foram
treinados no COCO, que tem a classe `boat`. O problema, portanto, nao e vocabulario.

A diferenca esta na distribuicao: no COCO, embarcacoes aparecem fotografadas de terra,
ocupando boa parte do quadro; aqui, a camera esta ao nivel do mar e o alvo ocupa dezenas
de pixels contra agua e ceu quase uniformes. E isso que o *fine-tuning* corrige.

In [ ]:
# ------------------------------------------------------------------
# A classe 'boat' existe nos modelos pre-treinados? (checagem)
# ------------------------------------------------------------------
print("=" * 70)
print("PROCURANDO A CLASSE 'boat' NOS MODELOS PRE-TREINADOS")
print("=" * 70)

for nome, m in modelos.items():
    classes = [c.lower() for c in m.names.values()]
    achou = "boat" in classes
    idx = [i for i, n in m.names.items() if n.lower() == "boat"]
    print(f"{nome:8s} ({len(classes)} classes): 'boat' presente = {achou}  (id={idx[0] if idx else 'n/d'})")

classes_ssd = [c.lower() for c in CLASSES_SSD]
achou = "boat" in classes_ssd
idx = [i for i, n in enumerate(CLASSES_SSD) if n.lower() == "boat"]
print(f"{'SSD300':8s} ({len(classes_ssd)} classes): 'boat' presente = {achou}  (id={idx[0] if idx else 'n/d'})")
print("=" * 70)

### Entao qual e o problema, se a classe existe?

A classe existe, mas o modelo foi treinado numa distribuicao diferente. No COCO, embarcacoes
aparecem fotografadas de terra ou de perto, ocupando boa parte do quadro; aqui a camera esta
ao nivel do mar e o alvo tipico ocupa poucas dezenas de pixels contra agua e ceu quase uniformes.

A celula abaixo conta, para cada imagem da sequencia, quantas caixas `boat` o YOLO11
pre-treinado encontra. O que interessa observar e quantas imagens ficam com zero caixas:
cada uma delas e um obstaculo real que o modelo generico simplesmente nao viu. E essa lacuna
que o *fine-tuning* fecha.

In [ ]:
# ------------------------------------------------------------------
# Quantas caixas 'boat' o YOLO11 encontra por imagem, na sequencia inteira?
# ------------------------------------------------------------------
modelo_sonda = modelos.get("YOLO11", list(modelos.values())[0])
contagens_boat = []
for caminho in caminhos_L_ordenados:
    rgb = carregar_rgb(caminho)
    dets = detectar_yolo(modelo_sonda, rgb, conf=CONF_MIN)
    contagens_boat.append(sum(1 for d in dets if d["classe"] == "boat"))

contagens_boat = np.array(contagens_boat)
print("=" * 70)
print(f"Imagens com 0 caixas 'boat'   : {(contagens_boat == 0).sum()}")
print(f"Imagens com 1 caixa  'boat'   : {(contagens_boat == 1).sum()}")
print(f"Imagens com 2+ caixas 'boat'  : {(contagens_boat >= 2).sum()}")
print(f"Maximo de caixas 'boat' numa unica imagem: {contagens_boat.max()}")
print("=" * 70)

plt.figure(figsize=(9, 3.5))
plt.hist(contagens_boat, bins=range(0, contagens_boat.max() + 2), align="left",
        color="#2a6ebb", edgecolor="white", rwidth=0.8)
plt.xlabel("Caixas 'boat' detectadas na imagem (YOLO11 pre-treinado)")
plt.ylabel("Numero de imagens")
plt.title("Ambiguidade do detector generico: quantas embarcacoes ele ve por imagem")
plt.tight_layout()
plt.savefig(SAIDAS / "ambiguidade_boat_generico.png", dpi=110, bbox_inches="tight")
plt.show()

print("\nCONCLUSAO: mesmo quando o modelo acerta a classe 'boat', ele nao sabe QUAL")
print("embarcacao interessa. Precisamos ensinar aos modelos o obstaculo especifica.")
print("-> transfer learning, com um dataset da proprio obstaculo.")

# Parte 5 — Anotacoes oficiais do MODD2

O MODD2 acompanha anotacoes feitas a mao e verificadas por especialista. Nao ha,
portanto, nenhum rotulo gerado automaticamente neste notebook: o *ground truth* e o
do proprio benchmark.

Cada quadro tem um `.mat` com dois campos:

| Campo | Conteudo |
|---|---|
| `obstacles` | matriz `(M, 4)` — uma linha por obstaculo, no formato `[x, y, largura, altura]` |
| `sea_edge`  | poligono da linha d'agua |

A celula abaixo converte `obstacles` para o formato YOLO (`classe cx cy w h`, normalizado)
e usa `sea_edge` para separar obstaculos grandes (a caixa cruza a linha d'agua) de
**pequenos** (inteiramente abaixo dela), a estratificacao nativa do conjunto, que funciona
como indicador indireto de distancia.

> Convencao de caixa. As caixas do MODD2 delimitam o obstaculo junto a linha d'agua e
>tendem a ser mais justas que as de um detector treinado no COCO, que costuma englobar toda a
>superestrutura da embarcacao. Isso deprime o IoU do modelo pre-treinado na avaliacao "antes".
>Por isso a avaliacao reporta tambem IoU ≥ 0,30 (`IOU_AVALIACAO_2`), alem do usual 0,50.

In [ ]:
# ------------------------------------------------------------------
# Leitura das anotacoes .mat do MODD2 -> caixas em pixels
# ------------------------------------------------------------------
import scipy.io as sio
import numpy as np
import cv2
import matplotlib.pyplot as plt

def ler_anotacao_modd2(caminho_mat):
    """Devolve (caixas_xyxy, sea_edge). Caixa = (x1, y1, x2, y2) em pixels."""
    a  = sio.loadmat(str(caminho_mat))["annotations"][0, 0]
    ob = a["obstacles"]
    se = a["sea_edge"]

    caixas = []
    if ob.size:
        # cast para float64 e obrigatorio: alguns arquivos vem em uint8 e
        # o produto largura*altura estoura silenciosamente
        for linha in np.atleast_2d(np.asarray(ob, dtype=np.float64)):
            x, y, w, h = linha[:4]
            if w > 0 and h > 0:
                caixas.append((float(x), float(y), float(x + w), float(y + h)))

    borda = np.asarray(se, dtype=np.float64) if se.size else None
    return caixas, borda

def altura_linha_dagua(borda, x):
    """y da linha d'agua na coluna x (interpolacao linear no poligono)."""
    if borda is None or len(borda) < 2:
        return None
    ordem = np.argsort(borda[:, 0])
    return float(np.interp(x, borda[ordem, 0], borda[ordem, 1]))

def classificar_obstaculo(caixa, borda):
    """'grande' se a caixa cruza a linha d'agua, 'pequeno' se fica toda abaixo."""
    x1, y1, x2, y2 = caixa
    y_agua = altura_linha_dagua(borda, (x1 + x2) / 2)
    if y_agua is None:
        return "indefinido"
    return "pequeno" if y1 >= y_agua else "grande"

# ---- Converte todas as anotacoes ----
anotacoes = {}          # caminho da imagem -> lista de caixas xyxy
estatisticas = []
for _, linha in df_imgs.iterrows():
    caixas, borda = ler_anotacao_modd2(linha["mat"])
    anotacoes[linha["caminho"]] = caixas
    for c in caixas:
        estatisticas.append({
            "sequencia": linha["sequencia"],
            "classe_tamanho": classificar_obstaculo(c, borda),
            "area_pct": 100.0 * (c[2] - c[0]) * (c[3] - c[1]) / (LARGURA_IMG * ALTURA_IMG),
        })

df_obj = pd.DataFrame(estatisticas)
n_vazios = sum(1 for v in anotacoes.values() if not v)

print("=" * 70)
print("ANOTACOES OFICIAIS DO MODD2")
print("=" * 70)
print(f"Quadros                        : {len(anotacoes)}")
print(f"  com ao menos um obstaculo    : {len(anotacoes) - n_vazios}")
print(f"  sem obstaculo (fundo puro)   : {n_vazios}")
print(f"Obstaculos anotados            : {len(df_obj)}")
if len(df_obj):
    print(f"  grandes (cruzam a linha)     : {(df_obj.classe_tamanho == 'grande').sum()}")
    print(f"  pequenos (abaixo da linha)   : {(df_obj.classe_tamanho == 'pequeno').sum()}")
    a = df_obj["area_pct"].values
    print(f"Area da caixa (% da imagem)    : mediana {np.median(a):.3f} | min {a.min():.4f} | max {a.max():.2f}")
    print(f"  caixas < 1% da imagem        : {(a < 1).sum()}/{len(a)} ({100*(a < 1).mean():.0f}%)")
print("=" * 70)

# ---- Verificacao visual obrigatoria: 4 quadros com o GT desenhado ----
com_obj = [c for c, v in anotacoes.items() if v]
amostra_vis = com_obj[:: max(1, len(com_obj) // 4)][:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, cam in zip(axes.ravel(), amostra_vis):
    rgb = carregar_rgb(cam)
    for (x1, y1, x2, y2) in anotacoes[cam]:
        cv2.rectangle(rgb, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 4)
    ax.imshow(rgb); ax.axis("off")
    ax.set_title(f"{Path(cam).name} — {len(anotacoes[cam])} obstaculo(s)", fontsize=10)
plt.suptitle("Ground truth oficial do MODD2 (vermelho) — confira antes de treinar", fontsize=14)
plt.tight_layout(); plt.savefig(SAIDAS / "gt_modd2.png", dpi=110, bbox_inches="tight"); plt.show()

## 6.2 — Particionando por sequencia (treino / validacao / teste)

Quadros vizinhos de uma mesma cena sao quase identicos. Se a particao fosse por quadro, o
modelo veria na validacao imagens praticamente iguais as do treino, e as metricas ficariam
infladas sem que ele tivesse generalizado nada.

Por isso a unidade de particao e a sequencia inteira: com 3 ou mais sequencias, o teste e
composto por cenas que o modelo nunca viu. Havendo menos de 3, o notebook cai para uma
particao por blocos de quadros consecutivos, estratificada por escala do alvo, e **avisa na
tela** que os conjuntos compartilham cena, ressalva que precisa ser declarada no artigo.

O conjunto de teste e usado uma unica vez, no fim. A validacao serve para escolher o
checkpoint; medir no mesmo conjunto que escolheu o modelo produz numero otimista.

In [ ]:
# ------------------------------------------------------------------
# Particao treino / validacao / teste
# ------------------------------------------------------------------
import random
import numpy as np

def _resumo(nome, quadros):
    n_obj = sum(len(anotacoes[c]) for c in quadros)
    print(f"  {nome:9s}: {len(quadros):5d} quadros | {n_obj:5d} obstaculos")

seqs = sorted(df_imgs["sequencia"].unique())
rng = random.Random(SEMENTE)

if len(seqs) >= 3:
    # ---- Particao POR SEQUENCIA: teste em cenas nunca vistas (protocolo forte) ----
    MODO_PARTICAO = "por sequencia"
    ordem = seqs[:]
    rng.shuffle(ordem)
    n_tr = max(1, round(len(ordem) * PROPORCAO_TREINO))
    n_va = max(1, round(len(ordem) * PROPORCAO_VALID))
    n_va = min(n_va, len(ordem) - n_tr - 1)          # garante >= 1 sequencia de teste
    grupos = {"train": ordem[:n_tr],
              "valid": ordem[n_tr:n_tr + n_va],
              "test":  ordem[n_tr + n_va:]}
    particao = {sp: list(df_imgs[df_imgs["sequencia"].isin(ss)]["caminho"])
                for sp, ss in grupos.items()}
else:
    # ---- Menos de 3 sequencias: particao por blocos de quadros consecutivos ----
    MODO_PARTICAO = "por blocos de quadros"
    grupos = {"train": [], "valid": [], "test": []}
    particao = {"train": [], "valid": [], "test": []}
    for seq in seqs:
        cam = list(df_imgs[df_imgs["sequencia"] == seq]["caminho"])
        blocos = [cam[i:i + TAMANHO_BLOCO] for i in range(0, len(cam), TAMANHO_BLOCO)]
        # ordena por escala mediana do alvo e intercala -> estratificacao por dificuldade
        def escala(bloco):
            ar = [ (c[2]-c[0])*(c[3]-c[1]) for q in bloco for c in anotacoes[q] ]
            return float(np.median(ar)) if ar else 0.0
        blocos.sort(key=escala)
        for i, b in enumerate(blocos):
            r = i % 20
            sp = "train" if r < 14 else ("valid" if r < 17 else "test")
            particao[sp].extend(b)
            grupos[sp].append(seq)

print("=" * 70)
print(f"PARTICAO — {MODO_PARTICAO}")
print("=" * 70)
if MODO_PARTICAO == "por sequencia":
    for sp in ("train", "valid", "test"):
        print(f"  {sp:9s}: {', '.join(grupos[sp])}")
    print("-" * 70)
for sp in ("train", "valid", "test"):
    _resumo(sp, particao[sp])
print("=" * 70)

if MODO_PARTICAO != "por sequencia":
    print("[aviso] Com menos de 3 sequencias nao ha particao por cena. Os conjuntos")
    print("        compartilham a mesma cena, e as metricas medem generalizacao para")
    print("        novos QUADROS, nao para novos CENARIOS. Declare isso no artigo.")

In [ ]:
# ------------------------------------------------------------------
# Montagem do dataset no formato YOLO (3 particoes)
# ------------------------------------------------------------------
import shutil

DIR_DATASET = DATASET / "modd2_yolo"
if DIR_DATASET.exists():
    shutil.rmtree(DIR_DATASET)
for sp in ("train", "valid", "test"):
    (DIR_DATASET / sp / "images").mkdir(parents=True, exist_ok=True)
    (DIR_DATASET / sp / "labels").mkdir(parents=True, exist_ok=True)

def salvar_label_yolo(caminho_txt, caixas, largura, altura):
    """Grava as caixas no formato YOLO. Lista vazia -> arquivo vazio (imagem negativa)."""
    linhas = []
    for (x1, y1, x2, y2) in caixas:
        cx = ((x1 + x2) / 2) / largura
        cy = ((y1 + y2) / 2) / altura
        w  = (x2 - x1) / largura
        h  = (y2 - y1) / altura
        if w <= 0 or h <= 0:
            continue
        cx, cy = min(max(cx, 0.0), 1.0), min(max(cy, 0.0), 1.0)
        w,  h  = min(w, 1.0), min(h, 1.0)
        linhas.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    Path(caminho_txt).write_text("\n".join(linhas) + ("\n" if linhas else ""))

# ATENCAO: sequencias diferentes reutilizam os mesmos numeros de quadro
# (1.597 colisoes no conjunto completo). Copiar para uma pasta plana usando
# apenas cam.name faria um arquivo sobrescrever o outro, perdendo imagens em
# silencio. Por isso o destino leva o nome da sequencia como prefixo.
seq_por_caminho = dict(zip(df_imgs["caminho"], df_imgs["sequencia"]))

contagem = {}
for sp, quadros in particao.items():
    for cam_str in quadros:
        cam = Path(cam_str)
        seq = seq_por_caminho[cam_str]
        base = f"{seq}__{cam.stem}"
        shutil.copy(cam, DIR_DATASET / sp / "images" / (base + cam.suffix))
        salvar_label_yolo(DIR_DATASET / sp / "labels" / (base + ".txt"),
                          anotacoes[cam_str], LARGURA_IMG, ALTURA_IMG)
    contagem[sp] = len(quadros)

# conferencia: o que foi copiado tem de bater com o que foi pedido
for sp in ("train", "valid", "test"):
    n_img = len(list((DIR_DATASET / sp / "images").glob("*.jp*g")))
    assert n_img == contagem[sp], (
        f"{sp}: {contagem[sp]} quadros pedidos mas {n_img} arquivos gravados "
        "-- colisao de nomes")

import yaml
YAML_DATASET = DIR_DATASET / "data.yaml"
cfg = {"path": str(DIR_DATASET), "train": "train/images", "val": "valid/images",
       "test": "test/images", "nc": 1, "names": [NOME_CLASSE]}
YAML_DATASET.write_text(yaml.safe_dump(cfg, sort_keys=False))

DIR_TREINO_IMG = DIR_DATASET / "train" / "images"
DIR_TREINO_LAB = DIR_DATASET / "train" / "labels"
DIR_VALID_IMG  = DIR_DATASET / "valid" / "images"
DIR_VALID_LAB  = DIR_DATASET / "valid" / "labels"
DIR_TESTE_IMG  = DIR_DATASET / "test"  / "images"
DIR_TESTE_LAB  = DIR_DATASET / "test"  / "labels"

print(f"[ok] Dataset YOLO pronto em {DIR_DATASET}")
print(f"     treino={contagem['train']}  validacao={contagem['valid']}  teste={contagem['test']}")
print(f"\ndata.yaml:\n{YAML_DATASET.read_text()}")

## 5.9 — Inspecionando o dataset antes de treinar

> Nunca treine sobre um dataset que voce nao olhou. Metade dos problemas de "o modelo nao aprende" sao caixas desalinhadas, labels vazios ou imagens corrompidas, e aqui, alem disso, ha o risco extra de a associacao automatica ter escolhido a embarcacao errada em algum ponto que passou pela revisao.

Esta celula mostra 8 imagens de treino com as caixas do ground truth desenhadas em verde, e imprime estatisticas importantes:

-**Total de imagens** por split
-**Objetos por imagem** (media), imagens negativas (sem alvo) sao esperadas e uteis: elas ensinam o modelo a nao alucinar um barco em qualquer lugar.
-**Distribuicao do tamanho das caixas** — se a maioria dos alvos rotulados for grande, o modelo treinado pode continuar falhando na embarcacao distante.

In [ ]:
# ------------------------------------------------------------------
# Visualizacao e estatisticas do dataset MODD2
# ------------------------------------------------------------------
def carregar_labels_yolo(caminho_label, largura, altura):
    """Le um .txt YOLO e devolve caixas em xyxy absoluto."""
    caixas = []
    p = Path(caminho_label)
    if not p.exists():
        return caixas
    for linha in p.read_text().strip().splitlines():
        partes = linha.split()
        if len(partes) < 5:
            continue
        _, cx, cy, w, h = [float(v) for v in partes[:5]]
        caixas.append(((cx - w/2) * largura, (cy - h/2) * altura,
                       (cx + w/2) * largura, (cy + h/2) * altura))
    return caixas

DIR_TREINO_IMG = DIR_DATASET / "train" / "images"
DIR_TREINO_LAB = DIR_DATASET / "train" / "labels"
DIR_VALID_IMG  = DIR_DATASET / "valid" / "images"
DIR_VALID_LAB  = DIR_DATASET / "valid" / "labels"

imgs_treino = sorted([p for p in DIR_TREINO_IMG.glob("*") if p.suffix.lower() in (".jpg", ".jpeg")])
imgs_valid  = sorted([p for p in DIR_VALID_IMG.glob("*")  if p.suffix.lower() in (".jpg", ".jpeg")])
imgs_teste  = sorted([p for p in DIR_TESTE_IMG.glob("*")  if p.suffix.lower() in (".jpg", ".jpeg")])

imgs_treino_com_alvo = [p for p in imgs_treino if carregar_labels_yolo(DIR_TREINO_LAB / (p.stem + ".txt"), 1, 1)]
amostra_visual = (imgs_treino_com_alvo or imgs_treino)[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, cam in zip(axes.ravel(), amostra_visual):
    img = cv2.cvtColor(cv2.imread(str(cam)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    caixas = carregar_labels_yolo(DIR_TREINO_LAB / (cam.stem + ".txt"), w, h)
    for (x1, y1, x2, y2) in caixas:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), max(2, w//300))
    ax.imshow(img); ax.set_title(f"{len(caixas)} alvo(s)", fontsize=11); ax.axis("off")
plt.suptitle("Amostras do dataset com o ground truth (verde)", fontsize=16)
plt.tight_layout(); plt.show()

# ---- Estatisticas ----
areas, por_imagem = [], []
for cam in imgs_treino:
    lab = DIR_TREINO_LAB / (cam.stem + ".txt")
    n = 0
    if lab.exists():
        for linha in lab.read_text().strip().splitlines():
            partes = linha.split()
            if len(partes) >= 5:
                areas.append(float(partes[3]) * float(partes[4]) * 100)  # % da area da imagem
                n += 1
    por_imagem.append(n)

print("=" * 70)
print(f"Imagens de treino    : {len(imgs_treino)}")
print(f"Imagens de validacao : {len(imgs_valid)}")
print(f"Objetos no treino    : {sum(por_imagem)}  (media {np.mean(por_imagem):.2f} por imagem)")
print(f"Imagens sem objeto (negativas) : {sum(1 for n in por_imagem if n == 0)}")
if areas:
    a = np.array(areas)
    print(f"Area das caixas (% da imagem): mediana {np.median(a):.2f}% | "
          f"min {a.min():.3f}% | max {a.max():.1f}%")
    print(f"Caixas 'pequenas' (<1% da imagem): {(a < 1).sum()} de {len(a)} ({(a<1).mean()*100:.0f}%)")
print("=" * 70)

if areas:
    plt.figure(figsize=(11, 4))
    plt.hist(np.clip(areas, 0, 20), bins=60, color="#2a6ebb", edgecolor="white")
    plt.xlabel("Area da caixa (% da imagem)"); plt.ylabel("Frequencia")
    plt.title("Distribuicao de escala do obstaculo — quanto mais massa a esquerda, mais alvos pequenos")
    plt.tight_layout(); plt.show()

# Parte 6 — Avaliacao: matriz de confusao ANTES do treinamento

## 6.1 — Como se mede um detector (e por que a matriz e 2×2)

Em classificacao, comparar predicao com verdade e trivial. Em deteccao nao e: o modelo pode acertar a classe e errar a posicao, ou achar o objeto certo duas vezes.

O protocolo padrao usa IoU (*Intersection over Union*):

```
IoU = area da intersecao / area da uniao
```

Com um limiar (usamos IoU ≥ 0.5), cada predicao vira:

| Situacao | Nome | Significado |
|---|---|---|
| Predicao casa com um GT | TP (verdadeiro positivo) | acertou o obstaculo |
| Predicao sem GT correspondente | FP (falso positivo) | achou um "boat" que nao e o alvo (ex.: embarcacao de fundo) |
| GT sem predicao correspondente | FN (falso negativo) | deixou passar o alvo |
| Fundo corretamente ignorado | *TN* | nao existe em deteccao |

O TN nao e contavel: uma imagem tem infinitas regioes de fundo que o modelo "corretamente nao detectou". Por isso a matriz de confusao de deteccao tem a celula inferior-direita marcada como N/A.

### As metricas derivadas

```
Precisao = TP / (TP + FP)      "do que eu falei, quanto era o alvo de verdade?"
Recall   = TP / (TP + FN)      "do que existia, quanto eu achei?"
F1       = 2·P·R / (P + R)     media harmonica das duas
```

### Por que nao precisamos de nenhum mapeamento de classes

A classe `boat` existe de verdade no COCO, entao usamos ela diretamente na avaliacao, sem truques. E esse e o proprio ponto desta parte: mesmo sem nenhum mapeamento, o modelo generico ainda vai errar bastante, porque qualquer `boat` (inclusive embarcacoes de fundo) conta como predicao candidata, e so uma delas, se alguma, coincide com o alvo real.

In [ ]:
# ------------------------------------------------------------------
# Funcoes de avaliacao: IoU, matching, matriz de confusao
# ------------------------------------------------------------------
def iou(caixa_a, caixa_b):
    ax1, ay1, ax2, ay2 = caixa_a
    bx1, by1, bx2, by2 = caixa_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return inter / (area_a + area_b - inter + 1e-9)

def casar_predicoes(preds, gts, limiar_iou=IOU_AVALIACAO):
    """Matching guloso por confianca. Devolve (TP, FP, FN)."""
    preds = sorted(preds, key=lambda d: -d["conf"])
    usados = set()
    tp = 0
    for p in preds:
        melhor, melhor_iou = -1, 0.0
        for j, g in enumerate(gts):
            if j in usados:
                continue
            v = iou(p["caixa"], g)
            if v > melhor_iou:
                melhor, melhor_iou = j, v
        if melhor >= 0 and melhor_iou >= limiar_iou:
            usados.add(melhor); tp += 1
    fp = len(preds) - tp
    fn = len(gts) - tp
    return tp, fp, fn

def avaliar_no_conjunto(funcao_detectar, imagens, dir_labels,
                        filtro_classes=None, limite=LIMITE_AVALIACAO, conf=CONF_MIN):
    """Roda o detector no conjunto de validacao e acumula TP/FP/FN."""
    TP = FP = FN = 0
    TP2 = FP2 = FN2 = 0
    # amostragem uniforme ao longo da sequencia (nao os N primeiros,
    # que seriam so o trecho inicial da captura)
    if limite and len(imagens) > limite:
        imagens = imagens[:: max(1, len(imagens) // limite)][:limite]
    for cam in imagens:
        bgr = cv2.imread(str(cam))
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        gts = carregar_labels_yolo(Path(dir_labels) / (cam.stem + ".txt"), w, h)
        preds = funcao_detectar(rgb)
        if filtro_classes is not None:
            preds = [p for p in preds if p["classe"].lower() in filtro_classes]
        tp, fp, fn = casar_predicoes(preds, gts)
        TP += tp; FP += fp; FN += fn
        tp2, fp2, fn2 = casar_predicoes(preds, gts, limiar_iou=IOU_AVALIACAO_2)
        TP2 += tp2; FP2 += fp2; FN2 += fn2
    return {"TP": TP, "FP": FP, "FN": FN,
            "f1@0.3":   2*TP2 / (2*TP2 + FP2 + FN2) if TP2 else 0.0,
            "precisao": TP / (TP + FP) if TP + FP else 0.0,
            "recall":   TP / (TP + FN) if TP + FN else 0.0,
            "f1":       2*TP / (2*TP + FP + FN) if TP else 0.0}

def plotar_matriz(ax, m, titulo):
    """Matriz 2x2 de deteccao: linhas = predito, colunas = real."""
    mat = np.array([[m["TP"], m["FP"]],
                    [m["FN"], 0]], dtype=float)
    im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=max(mat.max(), 1))
    rotulos = [["TP\n(acertou o obstaculo)", "FP\n(alarme falso)"],
               ["FN\n(perdeu o obstaculo)", "TN\n(nao aplicavel)"]]
    for i in range(2):
        for j in range(2):
            valor = "N/A" if (i == 1 and j == 1) else f"{int(mat[i, j])}"
            cor = "white" if mat[i, j] > mat.max() * 0.55 else "#123"
            ax.text(j, i, f"{rotulos[i][j]}\n\n{valor}", ha="center", va="center",
                    fontsize=11, color=cor, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["obstaculo (real)", "fundo (real)"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["obstaculo\n(predito)", "fundo\n(predito)"])
    ax.set_title(f'{titulo}\nP={m["precisao"]:.2f}  R={m["recall"]:.2f}  F1={m["f1"]:.2f}', fontsize=12)
    return im

print("Funcoes de avaliacao prontas.")

## 6.2 — Matriz de confusao dos modelos pre-treinados (ANTES)

Agora rodamos os quatro modelos COCO sobre o conjunto de validacao que acabamos de montar (Etapa 5.8/5.9) e comparamos as predicoes `boat` contra o rotulo `obstacle` que voce revisou.

**O que esperar:**
-**Recall razoavel** para achar "algum barco" — mas precisao baixa sempre que houver mais de uma embarcacao na imagem, porque o modelo generico nao sabe qual delas escolher
-O problema aparece mais em FP (barco errado contado como acerto) do que em recall bruto de "existe um barco"

Guarde esses numeros: eles sao o baseline contra o qual mediremos o ganho do fine-tuning.

 Avaliamos ate `LIMITE_AVALIACAO` imagens de validacao para manter o tempo razoavel.

In [ ]:
# ------------------------------------------------------------------
# Matriz de confusao ANTES do fine-tuning
# ------------------------------------------------------------------
FILTRO_BOAT = {"boat"}

metricas_antes = {}

for nome, m in modelos.items():
    print(f"Avaliando {nome} (pre-treinado)...", flush=True)
    metricas_antes[nome] = avaliar_no_conjunto(
        lambda rgb, mm=m: detectar_yolo(mm, rgb),
        imgs_teste, DIR_TESTE_LAB,
        filtro_classes=FILTRO_BOAT, limite=LIMITE_AVALIACAO)

print("Avaliando SSD300 (pre-treinado)...", flush=True)
metricas_antes["SSD300"] = avaliar_no_conjunto(
    lambda rgb: detectar_ssd(rgb),
    imgs_teste, DIR_TESTE_LAB,
    filtro_classes=FILTRO_BOAT, limite=LIMITE_AVALIACAO)

n = len(metricas_antes)
fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5))
axes = np.atleast_1d(axes)
for ax, (nome, m) in zip(axes, metricas_antes.items()):
    plotar_matriz(ax, m, f"{nome} — ANTES")
plt.suptitle("Matrizes de confusao ANTES do fine-tuning (modelos COCO, classe 'boat', IoU ≥ 0.5)",
             fontsize=15, y=1.04)
plt.tight_layout()
plt.savefig(SAIDAS / "matrizes_antes.png", dpi=120, bbox_inches="tight")
plt.show()

df_antes = pd.DataFrame(metricas_antes).T
df_antes[["precisao", "recall", "f1"]] = df_antes[["precisao", "recall", "f1"]].round(3)
print("\nMETRICAS ANTES DO FINE-TUNING:")
display(df_antes)

# Parte 7 — Fine-tuning dos quatro modelos

## 7.1 — Treinando YOLOv8, YOLO11 e YOLO26

O Ultralytics torna o fine-tuning quase invisivel:

```python
modelo = YOLO("yolo26n.pt")                 # carrega pesos COCO
modelo.train(data="data.yaml", epochs=60, imgsz=1024)
```

**O que acontece por baixo:**
1. Le o `data.yaml`, ve `nc: 1` e substitui a camada de saida (80 classes  1 classe: `obstacle`)
2. Mantem todos os outros pesos do COCO — e aqui que mora o transfer learning
3. Treina a rede inteira com learning rate baixo (`lr0=0.01` com warmup e cosine decay)
4. Aplica *data augmentation* automatico: mosaico, HSV, flip, escala, translacao

**Parametros que valem conhecer:**

| Parametro | Efeito |
|---|---|
| `patience=10` | early stopping se nao melhorar em 10 epocas |
| `mosaic` | junta 4 imagens em uma; otimo para objetos pequenos |
| `lr0` | learning rate inicial, baixo demais nao aprende, alto demais esquece o COCO |
| `imgsz=1024` | maior que o padrao (640); a embarcacao distante ocupa poucos pixels, entao resolucao maior preserva mais sinal |

 Tempo estimado: com `EPOCAS=20` e algumas centenas de imagens, espere algo entre 20 e 60 minutos por modelo em GPU T4 gratuita, os tres YOLO juntos podem passar de 2 horas. Se o tempo disponivel for curto, reduza `EPOCAS` na configuracao.

**O que observar durante o treino:** as colunas `box_loss` e `cls_loss` devem cair; `mAP50` deve subir. Se o mAP travar em zero, quase sempre e problema no dataset (caminhos errados ou labels vazios), nao no modelo.

In [ ]:
# ------------------------------------------------------------------
# Fine-tuning dos 3 modelos YOLO
# ------------------------------------------------------------------
modelos_treinados = {}

# Reaproveita exatamente os mesmos arquivos de pesos carregados na Etapa 3.1
# (inclui o fallback yolo12n.pt para o YOLO26, se foi o caso).
pesos_para_treino = [(nome, PESOS_USADOS[nome]) for nome in ("YOLOv8", "YOLO11", "YOLO26") if nome in PESOS_USADOS]

for nome, pesos in pesos_para_treino:
    if nome not in modelos:
        continue

    ckpt_drive = PASTA_DRIVE_CKPT / f"{nome}_best.pt"
    if ckpt_drive.exists():
        print(f"[ok] {nome}: checkpoint encontrado no Drive, pulando treino -> {ckpt_drive}")
        modelos_treinados[nome] = YOLO(str(ckpt_drive))
        continue

    print("\n" + "=" * 70)
    print(f"TREINANDO {nome}")
    print("=" * 70)
    m = YOLO(pesos)
    m.train(
        data=str(YAML_DATASET),
        epochs=EPOCAS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0 if DEVICE == "cuda" else "cpu",
        project=str(TREINOS),
        name=f"{nome}_modd2",
        exist_ok=True,
        patience=10,
        pretrained=True,     # <-- transfer learning: parte dos pesos COCO
        verbose=True,
        seed=SEMENTE,
    )
    melhor = TREINOS / f"{nome}_modd2" / "weights" / "best.pt"
    modelos_treinados[nome] = YOLO(str(melhor))
    print(f"[ok] {nome} treinado -> {melhor}")

    shutil.copy2(melhor, ckpt_drive)
    print(f"[drive] backup salvo em {ckpt_drive}")

print("\nModelos YOLO com fine-tuning:", list(modelos_treinados.keys()))

## 7.2 — Fine-tuning do SSD300 (na mao)

O `torchvision` nao tem uma API de alto nivel como o Ultralytics, entao fazemos o transfer learning explicitamente:

1. **Carregar o SSD300 com pesos COCO** — backbone VGG16 ja treinado
2. **Substituir a `classification_head`** por uma nova com `num_classes=2` (fundo + obstaculo). A `regression_head` (que preve coordenadas) e mantida: prever caixas e uma habilidade generica.
3. **Dataset PyTorch** que le os labels YOLO e converte para o formato do torchvision (`boxes` em xyxy absoluto, `labels` inteiros)
4. **Loop de treino** com SGD, momentum 0.9 e learning rate baixo (5e-3), decaimento cosseno

> Limitacao herdada do torchvision, nao do metodo: a implementacao padrao do SSD nao aceita imagens sem nenhuma caixa no lote de treino. Por isso o `Dataset` abaixo usa so as imagens com alvo rotulado, diferente dos YOLO (Ultralytics), que treinam com positivas e negativas juntas. Isso deve ser levado em conta ao comparar os quatro modelos no artigo, nao e uma escolha metodologica, e uma restricao da biblioteca.

 Com `EPOCAS=20`, espere de 10 a 30 minutos em T4, dependendo do numero de imagens positivas disponiveis.

In [ ]:
# ------------------------------------------------------------------
# Fine-tuning do SSD300: trocar a cabeca de classificacao e treinar
# ------------------------------------------------------------------
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection import _utils as det_utils

class DatasetMODD2(Dataset):
    """Le imagens + labels YOLO e devolve no formato esperado pelo torchvision.
    Usa apenas imagens com pelo menos uma caixa (limitacao do SSD do torchvision)."""
    def __init__(self, dir_imagens, dir_labels):
        self.itens = []
        for cam in sorted(Path(dir_imagens).glob("*")):
            if cam.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            lab = Path(dir_labels) / (cam.stem + ".txt")
            if lab.exists() and lab.read_text().strip():
                self.itens.append((cam, lab))

    def __len__(self):
        return len(self.itens)

    def __getitem__(self, i):
        cam, lab = self.itens[i]
        rgb = cv2.cvtColor(cv2.imread(str(cam)), cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        caixas = [c for c in carregar_labels_yolo(lab, w, h) if c[2] > c[0] + 1 and c[3] > c[1] + 1]
        if not caixas:
            caixas = [(0.0, 0.0, 1.0, 1.0)]
        tensor = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float() / 255.0
        alvo = {"boxes": torch.tensor(caixas, dtype=torch.float32),
                "labels": torch.ones((len(caixas),), dtype=torch.int64)}   # 1 = obstaculo_alvo
        return tensor, alvo

def juntar(lote):
    return tuple(zip(*lote))

ds_treino = DatasetMODD2(DIR_TREINO_IMG, DIR_TREINO_LAB)
ds_valid  = DatasetMODD2(DIR_VALID_IMG,  DIR_VALID_LAB)
dl_treino = DataLoader(ds_treino, batch_size=min(8, max(1, len(ds_treino))),
                       shuffle=True, collate_fn=juntar, num_workers=2)
print(f"SSD - imagens de treino (com alvo): {len(ds_treino)} | validacao (com alvo): {len(ds_valid)}")

# ---- Monta o SSD com cabeca nova (2 classes: fundo + obstaculo_alvo) ----
ssd_ajustado = ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1)
canais    = det_utils.retrieve_out_channels(ssd_ajustado.backbone, (300, 300))
n_ancoras = ssd_ajustado.anchor_generator.num_anchors_per_location()
ssd_ajustado.head.classification_head = SSDClassificationHead(
    in_channels=canais, num_anchors=n_ancoras, num_classes=2)
ssd_ajustado = ssd_ajustado.to(DEVICE)
print("Cabeca de classificacao substituida: 91 -> 2 classes (fundo, obstaculo_alvo)")

ckpt_ssd_drive = PASTA_DRIVE_CKPT / "ssd300_ajustado.pth"
if ckpt_ssd_drive.exists():
    print(f"[ok] SSD300: checkpoint encontrado no Drive, pulando treino -> {ckpt_ssd_drive}")
    ssd_ajustado.load_state_dict(torch.load(ckpt_ssd_drive, map_location=DEVICE))
    ssd_ajustado.eval()
    historico_ssd = []
else:
    parametros = [p for p in ssd_ajustado.parameters() if p.requires_grad]
    otimizador = torch.optim.SGD(parametros, lr=5e-3, momentum=0.9, weight_decay=5e-4)
    agendador  = torch.optim.lr_scheduler.CosineAnnealingLR(otimizador, T_max=EPOCAS)

    historico_ssd = []
    ssd_ajustado.train()
    for epoca in range(EPOCAS):
        soma, n_lotes = 0.0, 0
        for imagens, alvos in dl_treino:
            imagens = [i.to(DEVICE) for i in imagens]
            alvos   = [{k: v.to(DEVICE) for k, v in a.items()} for a in alvos]
            perdas = ssd_ajustado(imagens, alvos)
            total = sum(perdas.values())
            if not torch.isfinite(total):
                continue
            otimizador.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_(parametros, 10.0)
            otimizador.step()
            soma += float(total); n_lotes += 1
        agendador.step()
        media = soma / max(n_lotes, 1)
        historico_ssd.append(media)
        if (epoca + 1) % 5 == 0 or epoca == 0:
            print(f"  epoca {epoca+1:3d}/{EPOCAS}  loss = {media:.4f}")

    CAMINHO_SSD_AJUSTADO = TREINOS / "ssd300_modd2.pt"
    torch.save(ssd_ajustado.state_dict(), CAMINHO_SSD_AJUSTADO)
    ssd_ajustado.eval()
    print(f"\n[ok] SSD treinado e salvo em {CAMINHO_SSD_AJUSTADO}")

    shutil.copy2(CAMINHO_SSD_AJUSTADO, ckpt_ssd_drive)
    print(f"[drive] backup salvo em {ckpt_ssd_drive}")

if historico_ssd:
    plt.figure(figsize=(9, 3.5))
    plt.plot(range(1, len(historico_ssd) + 1), historico_ssd, marker="o", color="#c0392b", markersize=3)
    plt.xlabel("Epoca"); plt.ylabel("Loss media"); plt.title("Curva de treinamento do SSD300 fine-tuned")
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7.3 — Curvas de treino dos YOLO

O Ultralytics gera automaticamente um painel `results.png` por treino, com:
-**box_loss** — erro de localizacao das caixas
-**cls_loss** — erro de classificacao
-**dfl_loss** — erro de distribuicao das bordas (ausente no YOLO26, que removeu o DFL)
-**mAP50** e mAP50-95, as metricas oficiais de deteccao

Ele tambem salva a propria matriz de confusao do Ultralytics (`confusion_matrix_normalized.png`), calculada sobre o conjunto de validacao. Vale comparar com a nossa matriz manual, a logica e a mesma.

**Leitura das curvas:** losses descendo + mAP subindo = saudavel. mAP oscilando muito = learning rate alto ou dataset pequeno. Losses de validacao subindo enquanto as de treino caem = overfitting, hora de parar.

In [ ]:
# ------------------------------------------------------------------
# Paineis de resultados gerados pelo Ultralytics
# ------------------------------------------------------------------
from IPython.display import Image as ImagemIPy

for nome in modelos_treinados:
    pasta = TREINOS / f"{nome}_modd2"
    print("\n" + "=" * 70)
    print(f"RESULTADOS DO TREINO — {nome}")
    print("=" * 70)
    for arquivo in ["results.png", "confusion_matrix_normalized.png"]:
        p = pasta / arquivo
        if p.exists():
            display(ImagemIPy(filename=str(p), width=900))
        else:
            print(f"  ({arquivo} nao encontrado)")

# Parte 8 — Testando os modelos especialistas

## 8.1 — As mesmas duas imagens, agora com os modelos treinados

Voltamos exatamente as mesmas imagens de referencia da Parte 3 — `img_perto` e `img_longe`. Nada mudou na imagem; a unica variavel e o modelo.

Cada figura mostra os quatro modelos especialistas no obstaculo. Agora a classe detectada e literalmente `obstacle`, e (se o dataset e o treino foram bem feitos) so a embarcacao correta deveria aparecer marcada, mesmo quando ha outros barcos na cena.

**O que observar:**
- A embarcacao e detectada agora? Com que confianca?
- As outras embarcacoes, que antes apareciam como `boat` generico, deveriam parar de disparar deteccao positiva, essa e a diferenca central em relacao a Parte 3, e o efeito pratico de ensinar especificidade, nao vocabulario.
- Na imagem longe, comparar YOLO26 (com STAL para objetos pequenos) contra o SSD300 (que redimensiona para 300×300) costuma ser a diferenca mais gritante do notebook.

In [ ]:
# ------------------------------------------------------------------
# Imagem ALVO GRANDE com os modelos apos o fine-tuning
# ------------------------------------------------------------------
CLASSES_SSD_AJUSTADO = ["__background__", NOME_CLASSE]

def detectar_todos_depois(imagem_rgb, conf=CONF_MIN):
    import time
    res = {}
    for nome, m in modelos_treinados.items():
        t0 = time.time()
        res[nome] = {"det": detectar_yolo(m, imagem_rgb, conf=conf), "ms": (time.time()-t0)*1000}
    t0 = time.time()
    res["SSD300"] = {"det": detectar_ssd(imagem_rgb, conf=conf,
                                         modelo=ssd_ajustado, nomes_classes=CLASSES_SSD_AJUSTADO),
                     "ms": (time.time()-t0)*1000}
    return res

resultado_perto_depois = detectar_todos_depois(img_perto)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
for ax, (nome, r) in zip(axes.ravel(), resultado_perto_depois.items()):
    ax.imshow(desenhar_deteccoes(img_perto, r["det"]))
    ax.set_title(f'{nome} (fine-tuned) — {len(r["det"])} deteccao(oes) | {r["ms"]:.0f} ms', fontsize=14)
    ax.axis("off")
plt.suptitle(f" ALVO GRANDE — {NOME_PERTO} — modelos APOS o fine-tuning",
             fontsize=17, y=0.99)
plt.tight_layout()
plt.savefig(SAIDAS / "comparacao_perto_depois.png", dpi=110, bbox_inches="tight")
plt.show()

for nome, r in resultado_perto_depois.items():
    conf_max = max((d["conf"] for d in r["det"]), default=0)
    print(f"  {nome:8s}: {len(r['det'])} deteccao(oes) | confianca maxima {conf_max*100:.1f}%")

### Agora a imagem da alvo PEQUENO

A celula abaixo repete o mesmo procedimento na imagem dificil. Este e o teste decisivo: o modelo especialista consegue enxergar a embarcacao quando ela ocupa poucas dezenas de pixels, e continua ignorando as outras embarcacoes?

Compare o numero de deteccoes e a confianca maxima com o que os modelos COCO produziram na Parte 3.4.

In [ ]:
# ------------------------------------------------------------------
# Imagem ALVO PEQUENO com os modelos apos o fine-tuning
# ------------------------------------------------------------------
resultado_longe_depois = detectar_todos_depois(img_longe)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
for ax, (nome, r) in zip(axes.ravel(), resultado_longe_depois.items()):
    ax.imshow(desenhar_deteccoes(img_longe, r["det"]))
    ax.set_title(f'{nome} (fine-tuned) — {len(r["det"])} deteccao(oes) | {r["ms"]:.0f} ms', fontsize=14)
    ax.axis("off")
plt.suptitle(f" ALVO PEQUENO — {NOME_LONGE} — modelos APOS o fine-tuning",
             fontsize=17, y=0.99)
plt.tight_layout()
plt.savefig(SAIDAS / "comparacao_longe_depois.png", dpi=110, bbox_inches="tight")
plt.show()

for nome, r in resultado_longe_depois.items():
    conf_max = max((d["conf"] for d in r["det"]), default=0)
    print(f"  {nome:8s}: {len(r['det'])} deteccao(oes) | confianca maxima {conf_max*100:.1f}%")

## 8.2 — Antes × Depois lado a lado

Esta e a figura-sintese do notebook. Duas linhas por imagem:

-**Linha de cima:** modelo pre-treinado no COCO (marca qualquer `boat` que encontrar, alvo ou nao)
-**Linha de baixo:** o mesmo modelo apos o fine-tuning (deveria marcar so o obstaculo, com confianca alta)

E a mesma rede, a mesma imagem, os mesmos pesos de backbone. O que mudou foram algumas centenas de gradientes na cabeca de deteccao.

In [ ]:
# ------------------------------------------------------------------
# Painel comparativo ANTES x DEPOIS
# ------------------------------------------------------------------
def painel_antes_depois(imagem, antes, depois, titulo, arquivo):
    nomes = list(depois.keys())
    fig, axes = plt.subplots(2, len(nomes), figsize=(5.6*len(nomes), 9))
    axes = np.atleast_2d(axes)
    for col, nome in enumerate(nomes):
        det_a = antes.get(nome, {"det": []})["det"]
        axes[0, col].imshow(desenhar_deteccoes(imagem, det_a))
        axes[0, col].set_title(f"{nome} ANTES — {len(det_a)} obj", fontsize=12)
        axes[0, col].axis("off")
        det_d = depois[nome]["det"]
        axes[1, col].imshow(desenhar_deteccoes(imagem, det_d))
        axes[1, col].set_title(f"{nome} DEPOIS — {len(det_d)} deteccao(oes)", fontsize=12, color="#1a7a3a")
        axes[1, col].axis("off")
    plt.suptitle(titulo, fontsize=17, y=1.01)
    plt.tight_layout()
    plt.savefig(SAIDAS / arquivo, dpi=110, bbox_inches="tight")
    plt.show()

painel_antes_depois(img_perto, resultado_perto, resultado_perto_depois,
                    " ALVO GRANDE — pre-treinado (cima) × fine-tuned (baixo)",
                    "painel_perto.png")

painel_antes_depois(img_longe, resultado_longe, resultado_longe_depois,
                    " ALVO PEQUENO — pre-treinado (cima) × fine-tuned (baixo)",
                    "painel_longe.png")

## 8.3 — O conjunto completo com os modelos especialistas

Reprocessamos a mesma sequencia sintetica da Parte 4, agora com os quatro detectores especialistas. A comparacao e direta: mesmas imagens, mesma ordem, modelos diferentes.

**O que observar:**
- A caixa agora persiste ao longo da sequencia, em vez de piscar ou pular entre embarcacoes
- O rotulo e estavel (`obstacle`), sem alternar com outras deteccoes `boat`
- Quando a embarcacao fica muito pequena, mesmo o modelo especialista pode perde-la, o limite fisico de resolucao nao some com fine-tuning. A solucao para esse caso seria treinar com `imgsz` ainda maior ou usar *tiling* (recortar a imagem em blocos).

In [ ]:
# ------------------------------------------------------------------
# Anotando a sequencia com os modelos especialistas
# ------------------------------------------------------------------
videos_depois = {}

for nome, m in modelos_treinados.items():
    videos_depois[nome] = anotar_sequencia(caminhos_L_ordenados, SAIDAS / f"sequencia_{nome}_depois.mp4",
                                           lambda rgb, mm=m: detectar_yolo(mm, rgb),
                                           rotulo=f"{nome}-modd2")

videos_depois["SSD300"] = anotar_sequencia(
    caminhos_L_ordenados, SAIDAS / "sequencia_SSD300_depois.mp4",
    lambda rgb: detectar_ssd(rgb, modelo=ssd_ajustado, nomes_classes=CLASSES_SSD_AJUSTADO),
    rotulo="SSD300-modd2")

print("\nSequencias com os modelos especialistas prontas.")

### Assistindo as sequencias dos modelos especialistas

In [ ]:
# ------------------------------------------------------------------
# Exibindo as sequencias apos o fine-tuning
# ------------------------------------------------------------------
for nome, caminho in videos_depois.items():
    mostrar_video(caminho, largura=720, titulo=f" {nome} — APOS fine-tuning (detector do obstaculo)")

# Parte 9 — Avaliacao DEPOIS e o veredito

## 9.1 — Mesmo protocolo da Parte 7

Repetimos exatamente o mesmo protocolo da avaliacao ANTES:

-mesmo conjunto de teste
-mesmo limiar de IoU (0,5, com 0,3 como referencia secundaria)
-mesma confianca minima (0,25)
-mesmas funcoes de matching

A unica diferenca: nao ha mais filtro por `boat`, porque os modelos ajustados tem uma classe
unica e qualquer deteccao ja e candidata.

**Comparacao justa exige protocolo identico.** Mudar o limiar entre "antes" e "depois" faria
os numeros perderem sentido.

A celula seguinte acrescenta mAP@50, mAP@50-95 e tempo de inferencia (com aquecimento
e desvio-padrao), e grava `tabela_artigo.csv`.

In [ ]:
# ------------------------------------------------------------------
# Matriz de confusao DEPOIS do fine-tuning
# ------------------------------------------------------------------
metricas_depois = {}

for nome, m in modelos_treinados.items():
    print(f"Avaliando {nome} (fine-tuned)...", flush=True)
    metricas_depois[nome] = avaliar_no_conjunto(
        lambda rgb, mm=m: detectar_yolo(mm, rgb),
        imgs_teste, DIR_TESTE_LAB, filtro_classes=None, limite=LIMITE_AVALIACAO)

print("Avaliando SSD300 (fine-tuned)...", flush=True)
metricas_depois["SSD300"] = avaliar_no_conjunto(
    lambda rgb: detectar_ssd(rgb, modelo=ssd_ajustado, nomes_classes=CLASSES_SSD_AJUSTADO),
    imgs_teste, DIR_TESTE_LAB, filtro_classes=None, limite=LIMITE_AVALIACAO)

n = len(metricas_depois)
fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5))
axes = np.atleast_1d(axes)
for ax, (nome, m) in zip(axes, metricas_depois.items()):
    plotar_matriz(ax, m, f"{nome} — DEPOIS")
plt.suptitle("Matrizes de confusao APOS o fine-tuning (IoU ≥ 0.5)", fontsize=16, y=1.04)
plt.tight_layout()
plt.savefig(SAIDAS / "matrizes_depois.png", dpi=120, bbox_inches="tight")
plt.show()

df_depois = pd.DataFrame(metricas_depois).T
df_depois[["precisao", "recall", "f1"]] = df_depois[["precisao", "recall", "f1"]].round(3)
print("\nMETRICAS APOS O FINE-TUNING:")
display(df_depois)

In [ ]:
# ------------------------------------------------------------------
# mAP@50 / mAP@50-95 no conjunto de TESTE + tempo de inferencia
# ------------------------------------------------------------------
import time

def medir_latencia(fn_detectar, imagens, n_aquecimento=10, n_medidas=50):
    """Tempo por imagem, em ms. Descarta o aquecimento (compilacao/cache CUDA)."""
    amostra = imagens[:: max(1, len(imagens) // (n_aquecimento + n_medidas))]
    for cam in amostra[:n_aquecimento]:
        fn_detectar(carregar_rgb(cam))
    tempos = []
    for cam in amostra[n_aquecimento:n_aquecimento + n_medidas]:
        rgb = carregar_rgb(cam)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn_detectar(rgb)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        tempos.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(tempos)), float(np.std(tempos))

linhas_final = []
for nome, m in modelos_treinados.items():
    print(f"Avaliando {nome} (mAP no conjunto de teste)...", flush=True)
    r = m.val(data=str(YAML_DATASET), split="test", imgsz=IMGSZ,
              device=0 if DEVICE == "cuda" else "cpu", verbose=False)
    ms, sd = medir_latencia(lambda rgb, mm=m: detectar_yolo(mm, rgb), imgs_teste)
    linhas_final.append({
        "Modelo": nome,
        "Parametros (M)": round(sum(p.numel() for p in m.model.parameters()) / 1e6, 2),
        "mAP@50": round(float(r.box.map50), 3),
        "mAP@50-95": round(float(r.box.map), 3),
        "F1 (IoU 0.5)": round(metricas_depois[nome]["f1"], 3),
        "Tempo (ms)": f"{ms:.1f} ± {sd:.1f}",
    })

# O SSD do torchvision nao passa pelo .val() do Ultralytics: usa-se o F1 ja medido
ms, sd = medir_latencia(lambda rgb: detectar_ssd(rgb, modelo=ssd_ajustado,
                                                nomes_classes=CLASSES_SSD_AJUSTADO), imgs_teste)
linhas_final.append({
    "Modelo": "SSD300",
    "Parametros (M)": round(sum(p.numel() for p in ssd_ajustado.parameters()) / 1e6, 2),
    "mAP@50": "n/d", "mAP@50-95": "n/d",
    "F1 (IoU 0.5)": round(metricas_depois["SSD300"]["f1"], 3),
    "Tempo (ms)": f"{ms:.1f} ± {sd:.1f}",
})

tabela_artigo = pd.DataFrame(linhas_final)
print("\n" + "=" * 80)
print("TABELA DO ARTIGO — conjunto de TESTE")
print("=" * 80)
display(tabela_artigo)
tabela_artigo.to_csv(SAIDAS / "tabela_artigo.csv", index=False)
print(f"\nSalvo em {SAIDAS / 'tabela_artigo.csv'}")
print("\nAtencao: mAP do SSD marcado como n/d — o .val() do Ultralytics so aceita")
print("modelos YOLO. Para mAP do SSD seria preciso um avaliador COCO separado.")
sincronizar_saidas_com_drive()

## 9.2 — O painel comparativo completo

A figura abaixo empilha todas as oito matrizes: a linha de cima e o estado pre-treinado, a de baixo e o estado apos o fine-tuning. Logo em seguida, o grafico de barras compara precisao, recall e F1 par a par.

### Como ler o resultado

| Sintoma | Diagnostico | O que fazer |
|---|---|---|
| Recall subiu muito, precisao caiu | modelo virou "trigger-happy" | aumentar `CONF_MIN` |
| Precisao alta, recall baixo | conservador demais | baixar `CONF_MIN`, treinar mais epocas |
| Ambos baixos depois do treino | dataset pequeno demais, ou alvos pequenos demais para a resolucao usada | baixar PASSO_QUADROS, subir IMGSZ, mais epocas |
| FP alto so no SSD | resolucao 300x300 limitando o alvo pequeno | esperado; declarar como limitacao no artigo |

### O que a matriz nao te conta

-**Onde** o modelo erra (obstaculo pequeno? contra a costa? contra o reflexo do sol?)
-**Quao perto** as caixas estao do ground truth (isso e o mAP50-95)
-**Se a deteccao e confiavel de um ponto de vista so, ou dos dois** — e exatamente essa a pergunta da proxima secao.

In [ ]:
# ------------------------------------------------------------------
# Painel final: 8 matrizes + comparativo de metricas
# ------------------------------------------------------------------
nomes = [n for n in metricas_depois if n in metricas_antes]

fig, axes = plt.subplots(2, len(nomes), figsize=(5.2*len(nomes), 10))
axes = np.atleast_2d(axes)
for col, nome in enumerate(nomes):
    plotar_matriz(axes[0, col], metricas_antes[nome],  f"{nome} — ANTES (COCO/boat)")
    plotar_matriz(axes[1, col], metricas_depois[nome], f"{nome} — DEPOIS (fine-tuned)")
plt.suptitle("MATRIZES DE CONFUSAO — antes × depois do transfer learning", fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig(SAIDAS / "matrizes_antes_e_depois.png", dpi=120, bbox_inches="tight")
plt.show()

# ---- Barras comparativas ----
metricas_nomes = ["precisao", "recall", "f1"]
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
x = np.arange(len(nomes)); largura = 0.36
for ax, met in zip(axes, metricas_nomes):
    antes  = [metricas_antes[n][met]  for n in nomes]
    depois = [metricas_depois[n][met] for n in nomes]
    ax.bar(x - largura/2, antes,  largura, label="antes (boat generico)", color="#b0b7c3")
    ax.bar(x + largura/2, depois, largura, label="depois (obstaculo_alvo)", color="#2a9d5c")
    ax.set_xticks(x); ax.set_xticklabels(nomes, rotation=12)
    ax.set_ylim(0, 1.05); ax.set_title(met.upper(), fontsize=13)
    ax.grid(axis="y", alpha=0.3); ax.legend(fontsize=9)
    for i, (a, d) in enumerate(zip(antes, depois)):
        ax.text(i - largura/2, a + 0.02, f"{a:.2f}", ha="center", fontsize=9)
        ax.text(i + largura/2, d + 0.02, f"{d:.2f}", ha="center", fontsize=9, fontweight="bold")
plt.suptitle("Impacto do transfer learning nas metricas de deteccao", fontsize=16, y=1.05)
plt.tight_layout()
plt.savefig(SAIDAS / "comparativo_metricas.png", dpi=120, bbox_inches="tight")
plt.show()

# ---- Tabela consolidada ----
linhas = []
for nome in nomes:
    a, d = metricas_antes[nome], metricas_depois[nome]
    linhas.append({
        "Modelo": nome,
        "TP antes": a["TP"], "TP depois": d["TP"],
        "FP antes": a["FP"], "FP depois": d["FP"],
        "FN antes": a["FN"], "FN depois": d["FN"],
        "F1 antes": round(a["f1"], 3), "F1 depois": round(d["f1"], 3),
        "Ganho F1": f'{(d["f1"] - a["f1"]):+.3f}',
    })
tabela_final = pd.DataFrame(linhas)
print("\n" + "=" * 90)
print("QUADRO CONSOLIDADO — ANTES x DEPOIS")
print("=" * 90)
display(tabela_final)
tabela_final.to_csv(SAIDAS / "resultados_finais.csv", index=False)
print(f"\nSalvo em {SAIDAS/'resultados_finais.csv'}")

## Nota, a analise estereo foi removida

O MODD2 distribui anotacoes apenas para a camera esquerda. Sem rotulo no lado direito
nao ha como medir concordancia entre as duas vistas sem inventar caixas, o que reintroduziria
exatamente o tipo de rotulo automatico que este notebook passou a evitar.

A camera direita continua disponivel no conjunto e pode ser explorada em trabalho futuro,
seja para estimativa de distancia por disparidade, seja como mecanismo de reducao de falsos
alarmes por concordancia entre vistas.

### O que voce pode escrever no artigo a partir da Etapa 9.3

-**Redundancia por camera unica**: se a politica OU aumentar bastante a taxa de deteccao em relacao a uma unica camera, isso quantifica o ganho de ter duas vistas simultaneas, argumento direto para justificar (ou nao) um sistema estereo em vez de monocular.
-**Custo da redundancia**: compare esse ganho de recall com o aumento da taxa de falso alarme na politica OU — e o trade-off que a politica E resolve na direcao oposta.
-**Vies de camera**: o p-valor do teste de McNemar diz se uma das duas cameras é sistematicamente pior que a outra nesta configuracao (pode refletir foco, exposicao ou posicionamento no casco). Reporte o p-valor e qual camera concentrou mais falhas.
-**Degradacao com a distancia**: o grafico de discordancia por faixa de escala mostra se a concordancia entre cameras cai conforme o alvo fica pequeno, evidencia adicional, independente da matriz de confusao, de que a dificuldade do problema e dominada pela escala do alvo.
-**|Δconfianca|**: valores altos indicam que o modelo "quase" perdeu a deteccao em uma das vistas (confianca baixa) mesmo quando tecnicamente contou como acerto, um sinal de fragilidade que a matriz de confusao binaria (TP/FP/FN) nao capta sozinha.

Lembre-se de reportar, junto de qualquer numero desta secao, quantos pares ele foi calculado sobre (`n` nas tabelas acima), com poucas dezenas de pares, o teste de McNemar tem pouco poder estatistico, e isso deve ser declarado como limitacao no artigo.

# Conclusoes

O que ficou claro rodando isto:

**Modelo pre-treinado e generalista, e ter a classe nao basta.** Os quatro conhecem
`boat`. O problema nao e vocabulario, e dominio: camera ao nivel do mar, alvo de poucas
dezenas de pixels, fundo de agua e ceu quase uniformes, reflexo especular.

**Anotacao oficial muda a qualidade da conclusao.** Como o ground truth do MODD2 e
manual e verificado, nao corro o risco de ter rotulo gerado por um dos modelos que
estou comparando.

**A particao por sequencia e o que torna o numero honesto.** Testar em cena inedita
mede generalizacao. Testar em quadro vizinho mede memorizacao.

**Objeto pequeno e o eixo que separa as arquiteturas.** Com uns 92% dos obstaculos
abaixo de 1% da imagem, a resolucao de entrada deixa de ser detalhe de implementacao e
passa a decidir o resultado. E o que explica a posicao do SSD300, preso em 300x300.

**Acuracia sozinha nao decide o embarque.** A tabela final junta mAP e tempo de
inferencia porque a plataforma tem restricao de energia e de processamento. Vale
lembrar que a medicao de latencia aqui e em GPU de nuvem e variou entre sessoes, ou
seja, nao serve para escolher o modelo a embarcar sem repetir em hardware fixo.

# Parte 10 — Eficiência de dados (YOLO11)

Quantas imagens de treino bastam para o modelo estabilizar? Treina o YOLO11
(melhor modelo do artigo) com frações crescentes do conjunto de treino
(mesma validação/teste da Parte 6-9), mantendo época/lote/imgsz/semente
idênticos ao experimento principal.

**Não sobrescreve nada da Parte 6-9**: usa pastas (`modd2_yolo_ablacao`,
`treinos_ablacao`) e nomes de treino próprios. Os resultados publicados no
artigo (Tabela III) continuam intactos e reprodutíveis a partir das células
originais.

**IMPORTANTE — vai rodar sem supervisão**: rode a célula de montagem do
Drive abaixo AGORA (ela pede autorização interativa uma única vez) antes de
deixar o notebook treinando sozinho. Assim, ao final, os resultados são
copiados para o Drive automaticamente, mesmo que a sessão do Colab caia.

In [ ]:
# ------------------------------------------------------------------
# Monta o Drive AGORA (autorize interativamente antes de deixar rodando)
# So funciona depois de rodar o notebook completo desde o inicio
# (Etapa 1.3 em diante) -- e aqui so pra garantir Path mesmo se a
# ordem de execucao mudar.
# ------------------------------------------------------------------
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

PASTA_DRIVE_ABLACAO = Path("/content/drive/MyDrive/ablacao_eficiencia_dados")
PASTA_DRIVE_ABLACAO.mkdir(parents=True, exist_ok=True)
print(f"[ok] Drive montado. Resultados serao copiados para: {PASTA_DRIVE_ABLACAO}")

In [ ]:
# ------------------------------------------------------------------
# Treina o YOLO11 com fracoes crescentes do treino (25% / 50% / 75%)
# ------------------------------------------------------------------
import random

FRACOES = [0.25, 0.50, 0.75]   # o ponto de 100% reaproveita o YOLO11 ja treinado na Etapa 7
DATASET_ABLACAO = DATASET / "modd2_yolo_ablacao"
TREINOS_ABLACAO = RAIZ / "treinos_ablacao"

resultados_ablacao = []
rng_ablacao = random.Random(SEMENTE)
train_completo = particao["train"][:]   # mesma lista da particao principal, apenas lida

for frac in FRACOES:
    tag = f"frac{int(frac * 100)}"
    print("\n" + "=" * 70)
    print(f"ABLACAO {tag}: {frac:.0%} do treino ({round(len(train_completo) * frac)} de {len(train_completo)} imagens)")
    print("=" * 70)

    n = max(1, round(len(train_completo) * frac))
    subconjunto = rng_ablacao.sample(train_completo, n)

    dir_frac = DATASET_ABLACAO / tag
    if dir_frac.exists():
        shutil.rmtree(dir_frac)
    (dir_frac / "train" / "images").mkdir(parents=True, exist_ok=True)
    (dir_frac / "train" / "labels").mkdir(parents=True, exist_ok=True)
    (dir_frac / "valid").mkdir(parents=True, exist_ok=True)
    (dir_frac / "test").mkdir(parents=True, exist_ok=True)

    # subconjunto do treino: copia de verdade (arquivos novos, so os sorteados)
    for cam_str in subconjunto:
        cam = Path(cam_str)
        seq = seq_por_caminho[cam_str]
        base = f"{seq}__{cam.stem}"
        shutil.copy(cam, dir_frac / "train" / "images" / (base + cam.suffix))
        salvar_label_yolo(dir_frac / "train" / "labels" / (base + ".txt"),
                          anotacoes[cam_str], LARGURA_IMG, ALTURA_IMG)

    # valid/test: SEMPRE os mesmos da particao principal -> ligacao simbolica, sem copiar de novo
    for sp in ("valid", "test"):
        (dir_frac / sp / "images").symlink_to((DIR_DATASET / sp / "images").resolve())
        (dir_frac / sp / "labels").symlink_to((DIR_DATASET / sp / "labels").resolve())

    yaml_frac = dir_frac / "data.yaml"
    yaml_frac.write_text(yaml.safe_dump({
        "path": str(dir_frac), "train": "train/images", "val": "valid/images",
        "test": "test/images", "nc": 1, "names": [NOME_CLASSE],
    }, sort_keys=False))

    m = YOLO(PESOS_USADOS["YOLO11"])
    m.train(
        data=str(yaml_frac), epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH,
        device=0 if DEVICE == "cuda" else "cpu",
        project=str(TREINOS_ABLACAO), name=f"YOLO11_{tag}",
        exist_ok=True, patience=10, pretrained=True, verbose=False, seed=SEMENTE,
    )
    melhor = TREINOS_ABLACAO / f"YOLO11_{tag}" / "weights" / "best.pt"
    m_treinado = YOLO(str(melhor))
    r = m_treinado.val(data=str(yaml_frac), split="test", imgsz=IMGSZ,
                        device=0 if DEVICE == "cuda" else "cpu", verbose=False)

    resultados_ablacao.append({
        "fracao": frac,
        "n_imagens_treino": len(subconjunto),
        "mAP@50": round(float(r.box.map50), 3),
        "mAP@50-95": round(float(r.box.map), 3),
    })
    print(f"[ok] {tag}: {len(subconjunto)} imagens -> mAP@50={resultados_ablacao[-1]['mAP@50']}")

    # backup incremental no Drive, so o CSV parcial ate aqui (sobrevive a queda de sessao)
    pd.DataFrame(resultados_ablacao).to_csv(PASTA_DRIVE_ABLACAO / "tabela_ablacao_dados_PARCIAL.csv", index=False)

# ponto de 100%: reaproveita o YOLO11 ja treinado e avaliado na Parte 9 (nao retreina)
linha_100 = tabela_artigo[tabela_artigo["Modelo"] == "YOLO11"].iloc[0]
resultados_ablacao.append({
    "fracao": 1.00,
    "n_imagens_treino": len(train_completo),
    "mAP@50": float(linha_100["mAP@50"]),
    "mAP@50-95": float(linha_100["mAP@50-95"]),
})

tabela_ablacao = pd.DataFrame(resultados_ablacao).sort_values("fracao").reset_index(drop=True)
tabela_ablacao.to_csv(SAIDAS / "tabela_ablacao_dados.csv", index=False)
tabela_ablacao.to_csv(PASTA_DRIVE_ABLACAO / "tabela_ablacao_dados.csv", index=False)
print("\n" + "=" * 70)
print("EFICIENCIA DE DADOS — YOLO11")
print("=" * 70)
display(tabela_ablacao)
print(f"\nSalvo em {SAIDAS / 'tabela_ablacao_dados.csv'}")
print(f"Copiado para o Drive em {PASTA_DRIVE_ABLACAO / 'tabela_ablacao_dados.csv'}")

In [ ]:
# ------------------------------------------------------------------
# Grafico: mAP@50 x numero de imagens de treino
# ------------------------------------------------------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tabela_ablacao["n_imagens_treino"], tabela_ablacao["mAP@50"],
        marker="o", linewidth=2)
for _, row in tabela_ablacao.iterrows():
    ax.annotate(f"{row['fracao']:.0%}", (row["n_imagens_treino"], row["mAP@50"]),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax.set_xlabel("Imagens de treino")
ax.set_ylabel("mAP@50 (YOLO11, conjunto de teste)")
ax.set_title("Eficiência de dados")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(SAIDAS / "eficiencia_dados.png", dpi=150)
fig.savefig(PASTA_DRIVE_ABLACAO / "eficiencia_dados.png", dpi=150)
plt.show()
print(f"Salvo em {SAIDAS / 'eficiencia_dados.png'}")
print(f"Copiado para o Drive em {PASTA_DRIVE_ABLACAO / 'eficiencia_dados.png'}")

# Parte 11, mAP do SSD300 (avaliador próprio)

O `.val()` da Ultralytics só aceita modelos YOLO -- por isso o SSD300 ficava
de fora do mAP (Tabela III, marcado `n/d`) e da revocação por tamanho
(Tabela IV). Esta seção implementa Average Precision no padrão COCO
(interpolação de 101 pontos, classe única, IoU de 0.50 a 0.95) para avaliar
o SSD300 nas mesmas condições.

**Não recalcula o mAP dos modelos YOLO** -- os valores já publicados na
Tabela III vieram do avaliador oficial da Ultralytics e continuam como estão.

In [ ]:
# ------------------------------------------------------------------
# AP padrao COCO (classe unica) -- usado so para o SSD300
# ------------------------------------------------------------------
import numpy as np

def _average_precision(preds_todas, gts_por_imagem, limiar_iou):
    """AP no padrao COCO: classe unica, interpolacao de 101 pontos."""
    preds_todas = sorted(preds_todas, key=lambda d: -d["conf"])
    n_gt_total = sum(len(v) for v in gts_por_imagem.values())
    if n_gt_total == 0:
        return 0.0

    usados = {img: set() for img in gts_por_imagem}
    tp = np.zeros(len(preds_todas))
    fp = np.zeros(len(preds_todas))

    for i, p in enumerate(preds_todas):
        gts = gts_por_imagem.get(p["imagem"], [])
        melhor, melhor_iou = -1, 0.0
        for j, g in enumerate(gts):
            if j in usados[p["imagem"]]:
                continue
            v = iou(p["caixa"], g)
            if v > melhor_iou:
                melhor, melhor_iou = j, v
        if melhor >= 0 and melhor_iou >= limiar_iou:
            usados[p["imagem"]].add(melhor)
            tp[i] = 1
        else:
            fp[i] = 1

    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)
    recall = tp_cum / n_gt_total
    precisao = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)

    ap = 0.0
    for r in np.linspace(0, 1, 101):
        p_interp = precisao[recall >= r].max() if np.any(recall >= r) else 0.0
        ap += p_interp / 101
    return float(ap)


def avaliar_map(funcao_detectar, imagens, dir_labels, conf=0.001):
    """mAP@50 e mAP@50-95, para qualquer detector com a interface de
    detectar_yolo/detectar_ssd. conf baixo de proposito: o mAP precisa da
    curva completa de confianca, nao so das deteccoes acima do limiar
    operacional (CONF_MIN)."""
    preds_todas = []
    gts_por_imagem = {}
    for cam in imagens:
        rgb = carregar_rgb(cam)
        h, w = rgb.shape[:2]
        gts_por_imagem[str(cam)] = carregar_labels_yolo(Path(dir_labels) / (cam.stem + ".txt"), w, h)
        for d in funcao_detectar(rgb, conf=conf):
            preds_todas.append({**d, "imagem": str(cam)})

    ap50 = _average_precision(preds_todas, gts_por_imagem, 0.50)
    aps  = [_average_precision(preds_todas, gts_por_imagem, t) for t in np.arange(0.50, 1.00, 0.05)]
    return round(ap50, 3), round(float(np.mean(aps)), 3)


print("Calculando mAP do SSD300 (avaliador proprio, padrao COCO)...")
map50_ssd, map5095_ssd = avaliar_map(
    lambda rgb, conf: detectar_ssd(rgb, conf=conf, modelo=ssd_ajustado,
                                    nomes_classes=CLASSES_SSD_AJUSTADO),
    imgs_teste, DIR_TESTE_LAB)
print(f"SSD300 -> mAP@50={map50_ssd}  mAP@50-95={map5095_ssd}")

# ---- atualiza a linha do SSD300 na tabela do artigo (Tabela III) ----
idx_ssd = tabela_artigo.index[tabela_artigo["Modelo"] == "SSD300"][0]
tabela_artigo.loc[idx_ssd, "mAP@50"] = map50_ssd
tabela_artigo.loc[idx_ssd, "mAP@50-95"] = map5095_ssd
tabela_artigo.to_csv(SAIDAS / "tabela_artigo.csv", index=False)
display(tabela_artigo)
sincronizar_saidas_com_drive()

In [ ]:
# ------------------------------------------------------------------
# Revocacao do SSD300 por faixa de tamanho do alvo (para a Tabela IV)
# ------------------------------------------------------------------
# Mesmas faixas da Tabela IV do artigo: <0,05% / 0,05-0,5% / >0,5% da area
# da imagem, em IoU >= 0,5 (mesma convencao das demais linhas da tabela).

FAIXAS = [("<0,05%", 0, 0.05), ("0,05-0,5%", 0.05, 0.5), (">0,5%", 0.5, float("inf"))]

def revocacao_por_tamanho(funcao_detectar, imagens, dir_labels, conf=CONF_MIN, limiar_iou=IOU_AVALIACAO):
    contagem = {nome: [0, 0] for nome, _, _ in FAIXAS}  # [acertos, total]
    for cam in imagens:
        rgb = carregar_rgb(cam)
        h, w = rgb.shape[:2]
        gts = carregar_labels_yolo(Path(dir_labels) / (cam.stem + ".txt"), w, h)
        preds = sorted(funcao_detectar(rgb, conf=conf), key=lambda d: -d["conf"])
        usados = set()
        acertou = [False] * len(gts)
        for p in preds:
            melhor, melhor_iou = -1, 0.0
            for j, g in enumerate(gts):
                if j in usados:
                    continue
                v = iou(p["caixa"], g)
                if v > melhor_iou:
                    melhor, melhor_iou = j, v
            if melhor >= 0 and melhor_iou >= limiar_iou:
                usados.add(melhor)
                acertou[melhor] = True
        for g, ok in zip(gts, acertou):
            area_pct = 100.0 * (g[2] - g[0]) * (g[3] - g[1]) / (w * h)
            for nome, lo, hi in FAIXAS:
                if lo <= area_pct < hi:
                    contagem[nome][1] += 1
                    if ok:
                        contagem[nome][0] += 1
                    break
    return {nome: (round(m / t, 3) if t else float("nan")) for nome, (m, t) in contagem.items()}

rev_ssd = revocacao_por_tamanho(
    lambda rgb, conf: detectar_ssd(rgb, conf=conf, modelo=ssd_ajustado,
                                    nomes_classes=CLASSES_SSD_AJUSTADO),
    imgs_teste, DIR_TESTE_LAB)

print("SSD300 -- revocacao por faixa de tamanho (Tabela IV):")
for nome, valor in rev_ssd.items():
    print(f"  {nome:12s}: {valor}")

pd.Series(rev_ssd).to_csv(SAIDAS / "tabela4_ssd300.csv", header=["revocacao"])
print(f"\nSalvo em {SAIDAS / 'tabela4_ssd300.csv'}")
sincronizar_saidas_com_drive()

In [ ]:
# ------------------------------------------------------------------
# Empacotando todos os resultados para download
# ------------------------------------------------------------------
import shutil
from google.colab import files as colab_files

pacote = "/content/resultados_obstaculo"
shutil.make_archive(pacote, "zip", str(SAIDAS))
tamanho = os.path.getsize(pacote + ".zip") / 1e6

print("=" * 70)
print("ARQUIVOS GERADOS")
print("=" * 70)
for p in sorted(SAIDAS.iterdir()):
    print(f"  {p.name:45s} {p.stat().st_size/1e6:8.2f} MB")
print("=" * 70)
print(f"Pacote: {pacote}.zip ({tamanho:.1f} MB)")

# Descomente para baixar automaticamente:
# colab_files.download(pacote + ".zip")